In [ ]:
import os
import json
from pathlib import Path
import pandas as pd

harness_dir = Path("/content/drive/MyDrive/CyberDNA_LANL/cyberdna_validation_harness")
print("=== HARNESS DIRECTORY CONTENT ===")
if harness_dir.exists():
    for p in harness_dir.iterdir():
        print(f"{p.name} | Size: {p.stat().st_size / 1024:.2f} KB")
else:
    print("[-] Harness directory does not exist!")


=== HARNESS DIRECTORY CONTENT ===
validation_metrics.json | Size: 5.30 KB
confusion_matrices.json | Size: 1.36 KB
validation_report.md | Size: 2.14 KB


In [ ]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path

base_dir = Path('/content/drive/MyDrive/CyberDNA_LANL')
harness_dir = base_dir / 'cyberdna_validation_harness'

print('=== STEP 1: READING VALIDATION HARNESS ARTIFACTS ===')

metrics_file = harness_dir / 'validation_metrics.json'
matrices_file = harness_dir / 'confusion_matrices.json'
report_file = harness_dir / 'validation_report.md'

if metrics_file.exists():
    print(f'\n[+] Loading {metrics_file.name}:')
    with open(metrics_file, 'r') as f:
        metrics = json.load(f)
    print(json.dumps(metrics, indent=2))
else:
    print('[-] validation_metrics.json not found.')

if matrices_file.exists():
    print(f'\n[+] Loading {matrices_file.name}:')
    with open(matrices_file, 'r') as f:
        matrices = json.load(f)
    print(json.dumps(matrices, indent=2))
else:
    print('[-] confusion_matrices.json not found.')

if report_file.exists():
    print(f'\n[+] Reading {report_file.name} (First 1000 chars):')
    print(report_file.read_text()[:1000] + '\n... [TRUNCATED] ...')
else:
    print('[-] validation_report.md not found.')

=== STEP 1: READING VALIDATION HARNESS ARTIFACTS ===

[+] Loading validation_metrics.json:
{
  "A_User-Disjoint": {
    "high_failure_profile": {
      "precision": 1.0,
      "recall": 1.0,
      "f1-score": 1.0,
      "support": 1.0
    },
    "normal_activity": {
      "precision": 1.0,
      "recall": 1.0,
      "f1-score": 1.0,
      "support": 169830.0
    },
    "privileged_admin_profile": {
      "precision": 1.0,
      "recall": 1.0,
      "f1-score": 1.0,
      "support": 339.0
    },
    "accuracy": 1.0,
    "macro avg": {
      "precision": 1.0,
      "recall": 1.0,
      "f1-score": 1.0,
      "support": 170170.0
    },
    "weighted avg": {
      "precision": 1.0,
      "recall": 1.0,
      "f1-score": 1.0,
      "support": 170170.0
    }
  },
  "B_User-Disjoint": {
    "high_failure_profile": {
      "precision": 1.0,
      "recall": 1.0,
      "f1-score": 1.0,
      "support": 1.0
    },
    "normal_activity": {
      "precision": 1.0,
      "recall": 1.0,
      "f1-sco

### Part 1: Verification, Confusion Matrix Reconstruction & Ground-Truth Scan
We will scan our parquet telemetry to verify the temporal split boundaries (specifically looking at Hour 9) and audit why `high_failure_profile` holds exactly 1 sample in the User-Disjoint test set. We will also output a search trace verifying if any red-team annotations exist.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import json

base_dir = Path('/content/drive/MyDrive/CyberDNA_LANL')
data_path = base_dir / 'lanl_host_aware_aggregated.parquet'

print('=== PART 1.1: BOUNDARY ANALYSIS ===')
if data_path.exists():
    df_eval = pd.read_parquet(data_path)
    print(f'Loaded {len(df_eval):,} aggregated rows.')

    # Check exact hours in dataset
    hours = sorted(df_eval['hour_window'].unique())
    print('Unique Hours in Dataset:', hours)

    # Recalculate 75th percentile split hour
    split_hour = int(df_eval['hour_window'].quantile(0.75))
    print(f'Calculated 75th Percentile Hour: {split_hour}')

    train_hours = df_eval[df_eval['hour_window'] < split_hour]['hour_window'].unique()
    test_hours = df_eval[df_eval['hour_window'] >= split_hour]['hour_window'].unique()
    print(f'Training hours (strictly < {split_hour}): {sorted(train_hours)}')
    print(f'Testing hours (strictly >= {split_hour}): {sorted(test_hours)}')
    print(f'Does hour 9 belong to training? {9 in train_hours}')
else:
    print('[-] Parquet data missing.')

=== PART 1.1: BOUNDARY ANALYSIS ===
Loaded 541,209 aggregated rows.
Unique Hours in Dataset: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]
Calculated 75th Percentile Hour: 10
Training hours (strictly < 10): [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]
Testing hours (strictly >= 10): [np.int64(10), np.int64(11), np.int64(12)]
Does hour 9 belong to training? True


In [ ]:
print('=== PART 1.2: CLASS SUPPORT & DISJOINT-USER SAMPLE AUDIT ===')
if data_path.exists():
    # Standardize column naming
    df_eval = df_eval.rename(columns={'failed_logons': 'failed_ins' if 'failed_ins' in df_eval.columns else 'failed_logins'})
    df_eval['is_computer'] = df_eval['user'].str.endswith('$').astype(int)
    df_eval['total_auth_attempts'] = df_eval['successful_logons'] + df_eval['failed_logins']
    df_eval['login_failure_rate'] = np.where(df_eval['total_auth_attempts'] > 0, df_eval['failed_logins'] / df_eval['total_auth_attempts'], 0.0)

    # Generate behavior classes
    conds = [
        (df_eval['failed_logins'] >= 50) & (df_eval['login_failure_rate'] > 0.8),
        (df_eval['privilege_events'] >= 100) & (df_eval['is_computer'] == 0)
    ]
    df_eval['behavior_class'] = np.select(conds, ['high_failure_profile', 'privileged_admin_profile'], default='normal_activity')

    # Trace distribution of 'high_failure_profile'
    hf_users = df_eval[df_eval['behavior_class'] == 'high_failure_profile']['user'].unique()
    print(f'Total High Failure Profile rows: {len(df_eval[df_eval["behavior_class"] == "high_failure_profile"])}')
    print(f'Unique accounts exhibiting High Failure behavior: {len(hf_users)}')
    print('High Failure Accounts list:', hf_users)

    # Audit user-disjoint test index allocation
    from sklearn.model_selection import GroupShuffleSplit
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(df_eval, df_eval['behavior_class'], groups=df_eval['user']))

    test_df = df_eval.iloc[test_idx]
    print('\nUser-Disjoint Test Set Profile Counts:')
    print(test_df['behavior_class'].value_counts())
else:
    print('[-] Parquet data missing.')

=== PART 1.2: CLASS SUPPORT & DISJOINT-USER SAMPLE AUDIT ===
Total High Failure Profile rows: 223
Unique accounts exhibiting High Failure behavior: 16
High Failure Accounts list: ['Administrator' 'Comp065845$' 'User287926' 'User031784' 'User515356'
 'User937927' 'User015659' 'User006226' 'User506790' 'User986685'
 'User374774' 'User936492' 'User285859' 'User071989' 'User355833'
 'User944022']

User-Disjoint Test Set Profile Counts:
behavior_class
normal_activity             169830
privileged_admin_profile       339
high_failure_profile             1
Name: count, dtype: int64


### Part 2: Advanced Isolation Forest Ranking & Alert-Threshold Analysis
We will fit an Isolation Forest specifically on normal training activities, score our disjoint-user holdout set, and test its alert performance at defensible thresholds (e.g., 99th, 95th, 90th percentiles).

In [ ]:
from sklearn.ensemble import IsolationForest
import pandas as pd
import numpy as np

print('=== PART 2.1: FIT ISOLATION FOREST ===')
if data_path.exists():
    features_list = [
        'successful_logons', 'failed_logins', 'logoffs', 'explicit_credential_events',
        'privilege_events', 'process_starts', 'process_ends', 'kerberos_tgt_events',
        'kerberos_service_ticket_events', 'credential_validation_events',
        'is_computer', 'is_system', 'total_auth_attempts', 'login_failure_rate',
        'process_start_rate_per_success'
    ]

    # Build features sequence mapping
    df_eval['is_system'] = (df_eval['user'] == 'system').astype(int)
    df_eval['process_start_rate_per_success'] = np.where(df_eval['successful_logons'] > 0, df_eval['process_starts'] / df_eval['successful_logons'], df_eval['process_starts'])

    X = df_eval[features_list].fillna(0).astype(float)
    y = df_eval['behavior_class']

    # Isolate training fold normal samples
    train_normal_X = X.iloc[train_idx][y.iloc[train_idx] == 'normal_activity']

    # Fit Isolation Forest with no arbitrary hardcoded contamination
    clf_iforest = IsolationForest(contamination='auto', random_state=42, n_jobs=-1)
    clf_iforest.fit(train_normal_X)
    print('[+] Isolation Forest trained successfully.')

    # Score testing fold
    test_X = X.iloc[test_idx]
    test_y = y.iloc[test_idx]

    # Higher scores mean normal; lower scores mean abnormal
    anomaly_scores = clf_iforest.decision_function(test_X)

    # Map scores back to index
    test_results = pd.DataFrame({
        'true_class': test_y,
        'score': anomaly_scores,
        'user': df_eval.iloc[test_idx]['user'],
        'host': df_eval.iloc[test_idx]['host']
    })
else:
    print('[-] Parquet data missing.')

=== PART 2.1: FIT ISOLATION FOREST ===
[+] Isolation Forest trained successfully.


In [ ]:
print('=== PART 2.2: MULTI-THRESHOLD ALERT EVALUATION ===')
# Evaluate thresholds at top 1%, 5%, and 10% anomalies
for pct in [1, 5, 10]:
    threshold = np.percentile(test_results['score'], pct)
    test_results[f'alert_{pct}pct'] = (test_results['score'] <= threshold).astype(int)

    alerts = test_results[test_results[f'alert_{pct}pct'] == 1]
    print(f'\nThreshold at {pct}% anomaly window (Score <= {threshold:.4f}):')
    print(f'  - Total Alerts Generated: {len(alerts):,}')
    print('  - True Classes Flagged:')
    print(alerts['true_class'].value_counts())

    # Characterize flagged hosts and users
    unique_flagged_users = alerts['user'].nunique()
    unique_flagged_hosts = alerts['host'].nunique()
    print(f'  - Unique Flagged Users: {unique_flagged_users:,} | Unique Flagged Hosts: {unique_flagged_hosts:,}')

=== PART 2.2: MULTI-THRESHOLD ALERT EVALUATION ===

Threshold at 1% anomaly window (Score <= -0.0842):
  - Total Alerts Generated: 1,751
  - True Classes Flagged:
true_class
normal_activity             1412
privileged_admin_profile     338
high_failure_profile           1
Name: count, dtype: int64
  - Unique Flagged Users: 456 | Unique Flagged Hosts: 433

Threshold at 5% anomaly window (Score <= 0.0136):
  - Total Alerts Generated: 8,633
  - True Classes Flagged:
true_class
normal_activity             8293
privileged_admin_profile     339
high_failure_profile           1
Name: count, dtype: int64
  - Unique Flagged Users: 1,396 | Unique Flagged Hosts: 2,653

Threshold at 10% anomaly window (Score <= 0.0517):
  - Total Alerts Generated: 18,584
  - True Classes Flagged:
true_class
normal_activity             18244
privileged_admin_profile      339
high_failure_profile            1
Name: count, dtype: int64
  - Unique Flagged Users: 2,623 | Unique Flagged Hosts: 5,668


### Part 3: Save Advanced Report
We will save this secondary model audit to `/content/drive/MyDrive/CyberDNA_LANL/cyberdna_validation_harness/advanced_model_audit.json` to complete Phase 1.5 safely.

In [ ]:
advanced_metrics = {
    "recalculated_support_user_disjoint": test_df['behavior_class'].value_counts().to_dict(),
    "temporal_split_hour": int(split_hour),
    "hour_9_in_train": bool(9 in train_hours),
    "iforest_alerts_1pct": int((test_results['score'] <= np.percentile(test_results['score'], 1)).sum()),
    "iforest_alerts_5pct": int((test_results['score'] <= np.percentile(test_results['score'], 5)).sum()),
    "iforest_alerts_10pct": int((test_results['score'] <= np.percentile(test_results['score'], 10)).sum())
}

adv_audit_path = base_dir / 'cyberdna_validation_harness' / 'advanced_model_audit.json'
with open(adv_audit_path, 'w') as f:
    json.dump(advanced_metrics, f, indent=4)
print(f'[+] Advanced audit metrics successfully written to {adv_audit_path}')

[+] Advanced audit metrics successfully written to /content/drive/MyDrive/CyberDNA_LANL/cyberdna_validation_harness/advanced_model_audit.json


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive/CyberDNA_LANL")
OUT = ROOT / "cyberdna_v2_experiment"
OUT.mkdir(parents=True, exist_ok=True)

# Required columns for this experiment
required = {"user", "hour_window"}

# Prefer a DataFrame already loaded in the notebook.
df = None
df_name = None

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        cols = set(obj.columns)
        if required.issubset(cols) and len(obj) > 0:
            # Prefer the largest matching engineered dataset.
            if df is None or len(obj) > len(df):
                df = obj.copy()
                df_name = name

if df is not None:
    print(f"Using existing DataFrame: {df_name}")
    print(f"Rows: {len(df):,}")
else:
    # Search likely processed-data files, not every CSV in the folder.
    candidates = []
    keywords = ("feature", "window", "aggregate", "processed",
                "dataset", "engineered")

    for pattern in ("*.csv", "*.parquet"):
        for path in ROOT.rglob(pattern):
            if any(word in path.name.lower() for word in keywords):
                candidates.append(path)

    print("No matching DataFrame found in notebook memory.")
    print("Candidate processed files:")

    for path in candidates[:60]:
        print(path)

    raise RuntimeError(
        "No dataset was loaded automatically. Choose the correct "
        "engineered CSV/Parquet file from the list, load it into df, "
        "then rerun this cell."
    )

print("\nColumns:")
print(df.columns.tolist())

print("\nShape:", df.shape)
print("\nSample:")
display(df.head(3))


Using existing DataFrame: df_data
Rows: 541,209

Columns:
['user', 'host', 'hour_window', 'successful_logons', 'failed_logons', 'logoffs', 'explicit_credential_events', 'privilege_events', 'process_starts', 'process_ends', 'kerberos_tgt_events', 'kerberos_service_ticket_events', 'credential_validation_events', 'is_computer', 'total_auth_attempts', 'login_failure_rate', 'true_rule_label']

Shape: (541209, 17)

Sample:


,user,host,hour_window,successful_logons,failed_logons,logoffs,explicit_credential_events,privilege_events,process_starts,process_ends,kerberos_tgt_events,kerberos_service_ticket_events,credential_validation_events,is_computer,total_auth_attempts,login_failure_rate,true_rule_label
0,ActiveDirectory$,ActiveDirectory,0,10182.0,0.0,10215.0,0.0,8720.0,123.0,0.0,0.0,2.0,0.0,1,10182.0,0.0,normal_activity
1,Administrator,ActiveDirectory,0,10374.0,0.0,10376.0,0.0,0.0,0.0,0.0,186.0,2697.0,66766.0,0,10374.0,0.0,normal_activity
2,Administrator,Comp044849,0,338.0,0.0,294.0,13.0,279.0,0.0,0.0,0.0,0.0,0.0,0,338.0,0.0,privileged_admin_profile


In [ ]:

import json
import joblib
from datetime import datetime, timezone
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler

# ------------------------------------------------------------
# 1. Validate and prepare the data
# ------------------------------------------------------------
df = df.copy()

if not {"user", "hour_window"}.issubset(df.columns):
    raise ValueError("Dataset must contain user and hour_window.")

df["hour_window"] = pd.to_numeric(df["hour_window"], errors="coerce")
df = df.dropna(subset=["user", "hour_window"]).copy()
df["user"] = df["user"].astype(str)

# Use only features that exist in the audited dataset.
candidate_features = [
    "successful_logons",
    "failed_logins",
    "logoffs",
    "explicit_credential_events",
    "privilege_events",
    "process_starts",
    "process_ends",
    "kerberos_tgt_events",
    "kerberos_service_ticket_events",
    "credential_validation_events",
    "is_computer",
    "is_system",
    "total_auth_attempts",
    "login_failure_rate",
    "process_start_rate_per_success",
]

features = [
    c for c in candidate_features
    if c in df.columns and pd.api.types.is_numeric_dtype(df[c])
]

if len(features) < 5:
    raise ValueError(
        f"Only {len(features)} usable numeric features found: {features}. "
        "Check that df is the engineered feature dataset."
    )

# Recreate heuristic labels for analysis only.
# These labels are NOT independent attack ground truth.
for col in ["failed_logins", "login_failure_rate",
            "privilege_events", "is_computer"]:
    if col not in df.columns:
        df[col] = 0

df["audit_label"] = "normal_activity"

high_failure = (
    (df["failed_logins"].fillna(0) >= 50) &
    (df["login_failure_rate"].fillna(0) > 0.8)
)

privileged = (
    (df["privilege_events"].fillna(0) >= 100) &
    (df["is_computer"].fillna(0) == 0)
)

# Make the label assignment explicit and mutually exclusive.
df.loc[privileged, "audit_label"] = "privileged_admin_profile"
df.loc[high_failure, "audit_label"] = "high_failure_profile"

# Fixed chronological split from the audit.
train_all = df[df["hour_window"] < 10].copy()
test = df[df["hour_window"] >= 10].copy()

# Fit the anomaly model only on heuristic-normal training windows.
train_normal = train_all[
    train_all["audit_label"] == "normal_activity"
].copy()

if len(train_normal) < 100:
    raise ValueError(
        f"Only {len(train_normal)} normal training rows. "
        "Check the dataset and split."
    )

if len(test) == 0:
    raise ValueError("No test rows at hour_window >= 10.")

print("Features:", features)
print(f"Training-period rows: {len(train_all):,}")
print(f"Normal training rows: {len(train_normal):,}")
print(f"Test-period rows: {len(test):,}")
print("Training hours:",
      train_all["hour_window"].min(),
      "to", train_all["hour_window"].max())
print("Test hours:",
      test["hour_window"].min(),
      "to", test["hour_window"].max())

# ------------------------------------------------------------
# 2. Clean features and fit transformations on training only
# ------------------------------------------------------------
def numeric_matrix(frame):
    X = frame[features].apply(pd.to_numeric, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(0).clip(lower=0)
    return X.astype("float32")

X_train_raw = numeric_matrix(train_normal)
X_test_raw = numeric_matrix(test)

# Counts and rates are nonnegative. log1p reduces count skew.
X_train_log = np.log1p(X_train_raw)
X_test_log = np.log1p(X_test_raw)

scaler = RobustScaler()
X_train = scaler.fit_transform(X_train_log)
X_test = scaler.transform(X_test_log)

# Keep the fit size manageable for Colab.
# This sample is drawn from training-period normal activity only.
MAX_FIT_ROWS = 100_000
rng = np.random.default_rng(2026)

if len(X_train) > MAX_FIT_ROWS:
    fit_idx = np.sort(
        rng.choice(len(X_train), MAX_FIT_ROWS, replace=False)
    )
else:
    fit_idx = np.arange(len(X_train))

model = IsolationForest(
    n_estimators=200,
    max_samples="auto",
    contamination="auto",
    random_state=2026,
    n_jobs=-1,
)

model.fit(X_train[fit_idx])

# Higher score means more anomalous.
train_global = -model.decision_function(X_train)
test_global = -model.decision_function(X_test)

# ------------------------------------------------------------
# 3. Estimate each user's baseline from training data only
# ------------------------------------------------------------
# Work in log-feature space. For each user and feature, measure
# deviation from that user's training median using robust MAD.
train_users = train_normal["user"].to_numpy()
test_users = test["user"].to_numpy()

global_median = X_train_log.median()
global_mad = (X_train_log - global_median).abs().median()
global_scale = (1.4826 * global_mad).replace(0, 1e-6).fillna(1e-6)

train_local = np.zeros(len(train_normal), dtype="float64")
test_local = np.zeros(len(test), dtype="float64")

# Build user baselines from training-period normal rows only.
user_baselines = {}
for user, group in train_normal.groupby("user", sort=False):
    values = np.log1p(numeric_matrix(group))
    med = values.median()
    mad = (values - med).abs().median()
    scale = (1.4826 * mad).replace(0, 1e-6).fillna(1e-6)

    user_baselines[str(user)] = (
        med.to_numpy(dtype="float64"),
        scale.to_numpy(dtype="float64"),
        len(group),
    )

global_med_arr = global_median.to_numpy(dtype="float64")
global_scale_arr = global_scale.to_numpy(dtype="float64")

def user_deviation(frame):
    values = np.log1p(numeric_matrix(frame)).to_numpy(dtype="float64")
    users = frame["user"].astype(str).to_numpy()
    scores = np.zeros(len(frame), dtype="float64")

    for i, user in enumerate(users):
        baseline = user_baselines.get(user)

        # Avoid trusting a personal profile with too little history.
        if baseline is not None and baseline[2] >= 5:
            med, scale, _ = baseline
        else:
            med, scale = global_med_arr, global_scale_arr

        z = np.abs(values[i] - med) / scale
        z = np.clip(z, 0, 10)
        scores[i] = float(np.median(z))

    return scores

train_local = user_deviation(train_normal)
test_local = user_deviation(test)

# ------------------------------------------------------------
# 4. Convert both scores to percentiles based on training only
# ------------------------------------------------------------
def percentile_against_training(reference, values):
    reference = np.sort(np.asarray(reference, dtype="float64"))
    values = np.asarray(values, dtype="float64")
    return np.searchsorted(reference, values, side="right") / len(reference)

train_global_pct = percentile_against_training(
    train_global, train_global
)
test_global_pct = percentile_against_training(
    train_global, test_global
)

train_local_pct = percentile_against_training(
    train_local, train_local
)
test_local_pct = percentile_against_training(
    train_local, test_local
)

# Combined score: 60% overall anomaly + 40% personal deviation.
train_combined = 0.60 * train_global_pct + 0.40 * train_local_pct
test_combined = 0.60 * test_global_pct + 0.40 * test_local_pct

# Alert thresholds come only from training-normal scores.
thresholds = {
    "top_1pct": float(np.quantile(train_combined, 0.99)),
    "top_5pct": float(np.quantile(train_combined, 0.95)),
    "top_10pct": float(np.quantile(train_combined, 0.90)),
}

# ------------------------------------------------------------
# 5. Produce alert-threshold and test-analysis reports
# ------------------------------------------------------------
results = test.copy()
results["global_anomaly_score"] = test_global
results["global_anomaly_percentile"] = test_global_pct
results["user_deviation_score"] = test_local
results["user_deviation_percentile"] = test_local_pct
results["combined_anomaly_score"] = test_combined

threshold_rows = []
for name, cutoff in thresholds.items():
    flagged = results["combined_anomaly_score"] >= cutoff

    threshold_rows.append({
        "threshold": name,
        "training_score_cutoff": cutoff,
        "test_alerts": int(flagged.sum()),
        "test_alert_rate_pct": round(100 * flagged.mean(), 4),
        "unique_users_flagged": int(results.loc[flagged, "user"].nunique()),
        "unique_hosts_flagged": (
            int(results.loc[flagged, "host"].nunique())
            if "host" in results.columns else None
        ),
    })

threshold_table = pd.DataFrame(threshold_rows)

# The following counts compare against heuristic labels only.
# They must not be reported as attack-detection accuracy.
label_rows = []
for name, cutoff in thresholds.items():
    flagged = results["combined_anomaly_score"] >= cutoff
    counts = (
        results.loc[flagged, "audit_label"]
        .value_counts()
        .to_dict()
    )

    label_rows.append({
        "threshold": name,
        "flagged_normal_heuristic": counts.get("normal_activity", 0),
        "flagged_privileged_heuristic": counts.get(
            "privileged_admin_profile", 0
        ),
        "flagged_high_failure_heuristic": counts.get(
            "high_failure_profile", 0
        ),
    })

label_table = pd.DataFrame(label_rows)

# Highest-ranked test windows for manual investigation.
top_alerts = results.sort_values(
    "combined_anomaly_score", ascending=False
).head(500)

# ------------------------------------------------------------
# 6. Save a versioned experiment without touching v1
# ------------------------------------------------------------
threshold_table.to_csv(OUT / "threshold_table.csv", index=False)
label_table.to_csv(OUT / "heuristic_label_overlap.csv", index=False)
top_alerts.to_csv(OUT / "top_500_test_alerts.csv", index=False)

joblib.dump(
    {
        "model": model,
        "scaler": scaler,
        "features": features,
        "user_baselines": user_baselines,
        "thresholds": thresholds,
        "weights": {"global_anomaly": 0.60, "user_deviation": 0.40},
        "training_split": "hour_window < 10",
        "fit_population": "heuristic-normal training windows",
    },
    OUT / "cyberdna_v2_behavioral_baseline.joblib",
)

report = {
    "experiment": "Cyber DNA v2 behavioral baseline",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "rows": {
        "training_period": int(len(train_all)),
        "normal_training": int(len(train_normal)),
        "test_period": int(len(test)),
    },
    "split": {
        "train_condition": "hour_window < 10",
        "test_condition": "hour_window >= 10",
        "train_min_hour": float(train_all["hour_window"].min()),
        "train_max_hour": float(train_all["hour_window"].max()),
        "test_min_hour": float(test["hour_window"].min()),
        "test_max_hour": float(test["hour_window"].max()),
    },
    "features": features,
    "thresholds": thresholds,
    "limitations": [
        "Heuristic labels are not independent attack ground truth.",
        "Anomaly scores indicate unusual activity, not confirmed maliciousness.",
        "Thresholds are derived from training-normal scores.",
        "The 60/40 combination is an initial experiment, not a validated optimum.",
    ],
}

with open(OUT / "experiment_report.json", "w") as f:
    json.dump(report, f, indent=2)

print("\n=== Alert thresholds ===")
display(threshold_table)

print("\n=== Overlap with heuristic labels (NOT attack metrics) ===")
display(label_table)

print("\n=== Highest-ranked test windows ===")
display(
    top_alerts[
        [
            c for c in [
                "user", "host", "hour_window", "audit_label",
                "global_anomaly_percentile",
                "user_deviation_percentile",
                "combined_anomaly_score",
            ] if c in top_alerts.columns
        ]
    ].head(20)
)

print("\nSaved files to:", OUT)
print("Existing v1 model was not overwritten.")


Features: ['successful_logons', 'logoffs', 'explicit_credential_events', 'privilege_events', 'process_starts', 'process_ends', 'kerberos_tgt_events', 'kerberos_service_ticket_events', 'credential_validation_events', 'is_computer', 'total_auth_attempts', 'login_failure_rate']
Training-period rows: 399,249
Normal training rows: 398,907
Test-period rows: 141,960
Training hours: 0 to 9
Test hours: 10 to 12

=== Alert thresholds ===


,threshold,training_score_cutoff,test_alerts,test_alert_rate_pct,unique_users_flagged,unique_hosts_flagged
0,top_1pct,0.970164,1217,0.8573,783,195
1,top_5pct,0.913120,6213,4.3766,3037,1021
2,top_10pct,0.883257,12796,9.0138,5601,1719



=== Overlap with heuristic labels (NOT attack metrics) ===


,threshold,flagged_normal_heuristic,flagged_privileged_heuristic,flagged_high_failure_heuristic
0,top_1pct,1097,120,0
1,top_5pct,6036,177,0
2,top_10pct,12619,177,0



=== Highest-ranked test windows ===


,user,host,hour_window,audit_label,global_anomaly_percentile,user_deviation_percentile,combined_anomaly_score
460129,Scanner,ActiveDirectory,12,privileged_admin_profile,0.999990,0.999817,0.999921
454167,Administrator,ActiveDirectory,12,normal_activity,0.999857,0.999872,0.999863
289657,Administrator,ActiveDirectory,10,normal_activity,0.999857,0.999867,0.999861
311540,Administrator,ActiveDirectory,11,normal_activity,0.999857,0.999845,0.999852
289661,Administrator,Comp044849,10,privileged_admin_profile,0.999794,0.999820,0.999804
311542,Administrator,Comp044849,11,privileged_admin_profile,0.999794,0.999820,0.999804
311544,Administrator,Comp154659,11,privileged_admin_profile,0.999782,0.999830,0.999801
289664,Administrator,Comp154659,10,privileged_admin_profile,0.999779,0.999830,0.999799
454171,Administrator,Comp154659,12,privileged_admin_profile,0.999779,0.999830,0.999799
454169,Administrator,Comp044849,12,privileged_admin_profile,0.999779,0.999820,0.999795



Saved files to: /content/drive/MyDrive/CyberDNA_LANL/cyberdna_v2_experiment
Existing v1 model was not overwritten.


In [ ]:

# Cyber DNA v2 — diagnose missed high-failure profiles
import pandas as pd
import numpy as np

diagnostic = results.copy()

high = diagnostic[
    diagnostic["audit_label"] == "high_failure_profile"
].copy()

print("High-failure test profiles:", len(high))

if len(high) == 0:
    print("No high-failure profiles exist in this test split.")
else:
    print("\nHigh-failure score distribution:")
    display(
        high[
            [
                c for c in [
                    "user", "host", "hour_window",
                    "failed_logins", "login_failure_rate",
                    "total_auth_attempts", "privilege_events",
                    "global_anomaly_percentile",
                    "user_deviation_percentile",
                    "combined_anomaly_score",
                ]
                if c in high.columns
            ]
        ]
        .sort_values("combined_anomaly_score", ascending=False)
        .head(30)
    )

    print("\nWhere do high-failure profiles rank?")
    for pct in [0.01, 0.05, 0.10]:
        cutoff = diagnostic["combined_anomaly_score"].quantile(1 - pct)
        caught = (
            high["combined_anomaly_score"] >= cutoff
        ).sum()
        print(
            f"Top {pct:.0%}: {caught}/{len(high)} "
            f"({caught / len(high):.1%})"
        )

print("\nScore distribution by heuristic label:")
display(
    diagnostic.groupby("audit_label")[
        [
            "global_anomaly_percentile",
            "user_deviation_percentile",
            "combined_anomaly_score",
        ]
    ].agg(["count", "median", "mean", "max"]).round(4)
)

print("\nFeature list used by the model:")
print(features)

if "failed_logins" not in features:
    print(
        "\nWARNING: failed_logins is not a model input. "
        "Add it in the next controlled experiment."
    )


High-failure test profiles: 0
No high-failure profiles exist in this test split.

Score distribution by heuristic label:


global_anomaly_percentile                       \
                                             count  median    mean  max   
audit_label                                                               
normal_activity                             141783  0.5037  0.5032  1.0   
privileged_admin_profile                       177  0.9996  0.9982  1.0   

                         user_deviation_percentile                          \
                                             count  median    mean     max   
audit_label                                                                  
normal_activity                             141783  0.8161  0.8291  1.0000   
privileged_admin_profile                       177  0.9393  0.9211  0.9998   

                         combined_anomaly_score                          
                                          count  median    mean     max  
audit_label                                                              
normal_activity                          141783  0.6295  0.6336  0.9999  
privileged_admin_profile                    177  0.9755  0.9674  0.9999


Feature list used by the model:
['successful_logons', 'logoffs', 'explicit_credential_events', 'privilege_events', 'process_starts', 'process_ends', 'kerberos_tgt_events', 'kerberos_service_ticket_events', 'credential_validation_events', 'is_computer', 'total_auth_attempts', 'login_failure_rate']



In [ ]:
-
from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/CyberDNA_LANL")

print("=== Current dataframe ===")
print("Shape:", df.shape)

for col in df.columns:
    if any(term in col.lower() for term in ["fail", "auth", "login", "label"]):
        print(f"{col}:")
        print(df[col].describe().to_dict() if pd.api.types.is_numeric_dtype(df[col])
              else df[col].value_counts(dropna=False).head(10).to_dict())

if "failed_logins" in df.columns:
    print("\nfailed_logins nonzero rows:", (df["failed_logins"] > 0).sum())
    print("failed_logins total:", df["failed_logins"].sum())

print("\n=== Other dataframes still in memory ===")
for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame) and name != "df":
        cols = list(obj.columns)
        relevant = [
            c for c in cols
            if any(term in c.lower() for term in ["fail", "login", "auth"])
        ]
        if relevant:
            print(f"\n{name}: shape={obj.shape}")
            print("Relevant columns:", relevant)
            for c in relevant:
                if pd.api.types.is_numeric_dtype(obj[c]):
                    print(c, "nonzero rows:", int((obj[c].fillna(0) != 0).sum()))

print("\n=== Candidate processed files ===")
for pattern in ("*.csv", "*.parquet"):
    for path in ROOT.rglob(pattern):
        if any(word in path.name.lower() for word in
               ["feature", "window", "aggregate", "processed",
                "dataset", "engineered", "validation"]):
            try:
                if path.suffix == ".csv":
                    sample = pd.read_csv(path, nrows=5)
                    cols = sample.columns.tolist()
                else:
                    sample = pd.read_parquet(path)
                    cols = sample.columns.tolist()

                relevant = [
                    c for c in cols
                    if any(term in c.lower() for term in
                           ["fail", "login", "auth", "label"])
                ]

                if relevant:
                    print(f"\n{path}")
                    print("Relevant columns:", relevant)
            except Exception as e:
                print(f"Could not inspect {path.name}: {e}")


SyntaxError: invalid syntax (1866946339.py, line 1)

In [ ]:

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/CyberDNA_LANL")

print("=== Current dataframe ===")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

for col in df.columns:
    if any(word in col.lower() for word in ["fail", "login", "auth"]):
        print(f"\nColumn: {col}")
        if pd.api.types.is_numeric_dtype(df[col]):
            print(df[col].describe())
            print("Nonzero rows:", int((df[col].fillna(0) != 0).sum()))
        else:
            print(df[col].value_counts(dropna=False).head(10))

print("\n=== Processed files containing login/failure features ===")

for pattern in ("*.csv", "*.parquet"):
    for path in ROOT.rglob(pattern):
        if not any(word in path.name.lower() for
                   word in ["feature", "window", "aggregate",
                            "processed", "dataset", "engineered"]):
            continue

        try:
            if path.suffix == ".csv":
                sample = pd.read_csv(path, nrows=5)
            else:
                sample = pd.read_parquet(path).head(5)

            cols = sample.columns.tolist()
            relevant = [
                c for c in cols
                if any(word in c.lower()
                       for word in ["fail", "login", "auth"])
            ]

            if relevant:
                print("\nFile:", path)
                print("Relevant columns:", relevant)

        except Exception as e:
            print("Could not inspect:", path.name, "|", str(e))


=== Current dataframe ===
Shape: (541209, 19)
Columns: ['user', 'host', 'hour_window', 'successful_logons', 'failed_logons', 'logoffs', 'explicit_credential_events', 'privilege_events', 'process_starts', 'process_ends', 'kerberos_tgt_events', 'kerberos_service_ticket_events', 'credential_validation_events', 'is_computer', 'total_auth_attempts', 'login_failure_rate', 'true_rule_label', 'failed_logins', 'audit_label']

Column: failed_logons
count    541209.000000
mean          0.342243
std          33.486277
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max        6458.000000
Name: failed_logons, dtype: float64
Nonzero rows: 4612

Column: total_auth_attempts
count    541209.000000
mean         13.985022
std         401.361955
min           0.000000
25%           1.000000
50%           1.000000
75%           9.000000
max      126280.000000
Name: total_auth_attempts, dtype: float64
Nonzero rows: 421455

Column: login_failure_rate
count    54120

In [ ]:
import pandas as pd
from pathlib import Path

print("=== 1. Label distributions ===")

for col in ["true_rule_label", "audit_label"]:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).to_string())

print("\n=== 2. Failed logons by label ===")

for col in ["true_rule_label", "audit_label"]:
    print(f"\nGrouped by {col}:")
    print(
        df.groupby(col, dropna=False)["failed_logons"]
        .agg(["count", "mean", "median", "max"])
        .to_string()
    )

print("\n=== 3. High-failure profiles ===")

for threshold in [10, 50, 100]:
    mask = (
        (df["failed_logons"] >= threshold)
        & (df["login_failure_rate"] > 0.8)
    )
    print(f"Threshold >= {threshold}: {int(mask.sum())} rows")

print("\n=== 4. Labels among high-failure profiles ===")

mask = (
    (df["failed_logons"] >= 50)
    & (df["login_failure_rate"] > 0.8)
)

for col in ["true_rule_label", "audit_label"]:
    print(f"\n{col}:")
    print(df.loc[mask, col].value_counts(dropna=False).to_string())

print("\n=== 5. Compare the two feature files ===")

root = Path("/content/drive/MyDrive/CyberDNA_LANL")

for filename in [
    "lanl_features.parquet",
    "lanl_host_aware_aggregated.parquet",
]:
    path = root / filename
    if not path.exists():
        print(f"\nMissing file: {filename}")
        continue

    sample = pd.read_parquet(path)
    print(f"\n{filename}")
    print("Shape:", sample.shape)
    print("Columns:", sample.columns.tolist())

    for col in [
        "failed_logons", "failed_logins",
        "login_failure_rate", "true_rule_label", "audit_label"
    ]:
        if col in sample.columns:
            print(f"\n{col}:")
            if pd.api.types.is_numeric_dtype(sample[col]):
                print("Nonzero rows:", int((sample[col].fillna(0) != 0).sum()))
                print("Maximum:", sample[col].max())
            else:
                print(sample[col].value_counts(dropna=False).head(10).to_string())

=== 1. Label distributions ===

true_rule_label:
true_rule_label
normal_activity             540467
privileged_admin_profile       519
high_failure_profile           223

audit_label:
audit_label
normal_activity             540690
privileged_admin_profile       519

=== 2. Failed logons by label ===

Grouped by true_rule_label:
                           count        mean  median     max
true_rule_label                                             
high_failure_profile         223  558.089686    87.0  6458.0
normal_activity           540467    0.061406     0.0   226.0
privileged_admin_profile     519   53.146435     0.0  2245.0

Grouped by audit_label:
                           count       mean  median     max
audit_label                                                
normal_activity           540690   0.291557     0.0  6458.0
privileged_admin_profile     519  53.146435     0.0  2245.0

=== 3. High-failure profiles ===
Threshold >= 10: 532 rows
Threshold >= 50: 223 rows
Threshold >= 1

In [ ]:
import pandas as pd
from pathlib import Path

# Work on a separate copy
df_v2 = df.copy()

# Remove the broken column created by the earlier fallback.
# Keep the genuine failed_logons feature.
df_v2 = df_v2.drop(columns=["failed_logins"], errors="ignore")

required = [
    "user", "host", "hour_window",
    "failed_logons", "login_failure_rate",
    "privilege_events", "is_computer",
    "true_rule_label"
]

missing = [c for c in required if c not in df_v2.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Confirm that the real failure feature is numeric.
df_v2["failed_logons"] = pd.to_numeric(
    df_v2["failed_logons"], errors="raise"
)

# Keep the previously defined temporal split.
train_mask = df_v2["hour_window"] < 10
test_mask = df_v2["hour_window"] >= 10

print("=== Dataset ===")
print("Rows:", len(df_v2))
print("Features include failed_logons:", "failed_logons" in df_v2.columns)
print("Failed-logon rows:", int((df_v2["failed_logons"] > 0).sum()))

print("\n=== Training period: hours < 10 ===")
print("Rows:", int(train_mask.sum()))
print(df_v2.loc[train_mask, "true_rule_label"].value_counts())

print("\n=== Test period: hours >= 10 ===")
print("Rows:", int(test_mask.sum()))
print(df_v2.loc[test_mask, "true_rule_label"].value_counts())

print("\n=== High-failure profiles by period ===")
for name, mask in [("Train", train_mask), ("Test", test_mask)]:
    part = df_v2.loc[mask]
    high = part["failed_logons"] >= 50

    print(f"\n{name}:")
    print("Rows with failed_logons >= 50:", int(high.sum()))
    print("Labels among these rows:")
    print(part.loc[high, "true_rule_label"].value_counts())

print("\n=== Label conflict ===")
print(pd.crosstab(
    df_v2["true_rule_label"],
    df["audit_label"],
    rownames=["true_rule_label"],
    colnames=["audit_label"],
    dropna=False
))

=== Dataset ===
Rows: 541209
Features include failed_logons: True
Failed-logon rows: 4612

=== Training period: hours < 10 ===
Rows: 399249
true_rule_label
normal_activity             398752
privileged_admin_profile       342
high_failure_profile           155
Name: count, dtype: int64

=== Test period: hours >= 10 ===
Rows: 141960
true_rule_label
normal_activity             141715
privileged_admin_profile       177
high_failure_profile            68
Name: count, dtype: int64

=== High-failure profiles by period ===

Train:
Rows with failed_logons >= 50: 258
Labels among these rows:
true_rule_label
high_failure_profile        155
normal_activity              93
privileged_admin_profile     10
Name: count, dtype: int64

Test:
Rows with failed_logons >= 50: 108
Labels among these rows:
true_rule_label
high_failure_profile        68
normal_activity             37
privileged_admin_profile     3
Name: count, dtype: int64

=== Label conflict ===
audit_label               normal_activity  pri

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/CyberDNA_LANL")
EXP = ROOT / "cyberdna_v2_experiment"

print("=== Experiment files ===")
for p in sorted(EXP.glob("*")):
    print(p.name)

print("\n=== Find the old feature-selection code ===")
for p in ROOT.rglob("*.py"):
    if "validation_harness" in str(p):
        continue
    try:
        content = p.read_text(errors="ignore")
        if "login_failure_rate" in content and (
            "IsolationForest" in content or "FEATURES" in content
        ):
            print(f"\n--- {p} ---")
            for i, line in enumerate(content.splitlines(), 1):
                if any(k in line for k in [
                    "FEATURES", "feature_cols", "failed_logins",
                    "failed_logons", "IsolationForest",
                    "fit(", "predict("
                ]):
                    print(f"{i}: {line[:200]}")
    except Exception:
        pass

=== Experiment files ===
cyberdna_v2_behavioral_baseline.joblib
experiment_report.json
heuristic_label_overlap.csv
threshold_table.csv
top_500_test_alerts.csv

=== Find the old feature-selection code ===


In [ ]:
import pandas as pd
from pathlib import Path

print("=== 1. Label distributions ===")

for col in ["true_rule_label", "audit_label"]:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).to_string())

print("\n=== 2. Failed logons by label ===")

for col in ["true_rule_label", "audit_label"]:
    print(f"\nGrouped by {col}:")
    print(
        df.groupby(col, dropna=False)["failed_logons"]
        .agg(["count", "mean", "median", "max"])
        .to_string()
    )

print("\n=== 3. High-failure profiles ===")

for threshold in [10, 50, 100]:
    mask = (
        (df["failed_logons"] >= threshold)
        & (df["login_failure_rate"] > 0.8)
    )
    print(f"Threshold >= {threshold}: {int(mask.sum())} rows")

print("\n=== 4. Labels among high-failure profiles ===")

mask = (
    (df["failed_logons"] >= 50)
    & (df["login_failure_rate"] > 0.8)
)

for col in ["true_rule_label", "audit_label"]:
    print(f"\n{col}:")
    print(df.loc[mask, col].value_counts(dropna=False).to_string())

print("\n=== 5. Compare the two feature files ===")

root = Path("/content/drive/MyDrive/CyberDNA_LANL")

for filename in [
    "lanl_features.parquet",
    "lanl_host_aware_aggregated.parquet",
]:
    path = root / filename
    if not path.exists():
        print(f"\nMissing file: {filename}")
        continue

    sample = pd.read_parquet(path)
    print(f"\n{filename}")
    print("Shape:", sample.shape)
    print("Columns:", sample.columns.tolist())

    for col in [
        "failed_logons", "failed_logins",
        "login_failure_rate", "true_rule_label", "audit_label"
    ]:
        if col in sample.columns:
            print(f"\n{col}:")
            if pd.api.types.is_numeric_dtype(sample[col]):
                print("Nonzero rows:", int((sample[col].fillna(0) != 0).sum()))
                print("Maximum:", sample[col].max())
            else:
                print(sample[col].value_counts(dropna=False).head(10).to_string())

=== 1. Label distributions ===

true_rule_label:
true_rule_label
normal_activity             540467
privileged_admin_profile       519
high_failure_profile           223

audit_label:
audit_label
normal_activity             540690
privileged_admin_profile       519

=== 2. Failed logons by label ===

Grouped by true_rule_label:
                           count        mean  median     max
true_rule_label                                             
high_failure_profile         223  558.089686    87.0  6458.0
normal_activity           540467    0.061406     0.0   226.0
privileged_admin_profile     519   53.146435     0.0  2245.0

Grouped by audit_label:
                           count       mean  median     max
audit_label                                                
normal_activity           540690   0.291557     0.0  6458.0
privileged_admin_profile     519  53.146435     0.0  2245.0

=== 3. High-failure profiles ===
Threshold >= 10: 532 rows
Threshold >= 50: 223 rows
Threshold >= 1

In [ ]:
import joblib
import json
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/CyberDNA_LANL")
EXP = ROOT / "cyberdna_v2_experiment"

model_path = EXP / "cyberdna_v2_behavioral_baseline.joblib"
report_path = EXP / "experiment_report.json"

print("=== SAVED MODEL ===")
obj = joblib.load(model_path)
print("Object type:", type(obj).__name__)

if hasattr(obj, "get_params"):
    print("Model parameters:", obj.get_params())

if hasattr(obj, "feature_names_in_"):
    print("Model input features:", list(obj.feature_names_in_))

if hasattr(obj, "named_steps"):
    print("Pipeline steps:", list(obj.named_steps.keys()))
    for name, step in obj.named_steps.items():
        if hasattr(step, "feature_names_in_"):
            print(f"{name} input features:", list(step.feature_names_in_))

print("\n=== EXPERIMENT REPORT ===")
with open(report_path, "r") as f:
    report = json.load(f)

print(json.dumps(report, indent=2, default=str)[:12000])

=== SAVED MODEL ===
Object type: dict

=== EXPERIMENT REPORT ===
{
  "experiment": "Cyber DNA v2 behavioral baseline",
  "created_utc": "2026-10-21T18:12:39.292285+00:00",
  "rows": {
    "training_period": 399249,
    "normal_training": 398907,
    "test_period": 141960
  },
  "split": {
    "train_condition": "hour_window < 10",
    "test_condition": "hour_window >= 10",
    "train_min_hour": 0.0,
    "train_max_hour": 9.0,
    "test_min_hour": 10.0,
    "test_max_hour": 12.0
  },
  "features": [
    "successful_logons",
    "logoffs",
    "explicit_credential_events",
    "privilege_events",
    "process_starts",
    "process_ends",
    "kerberos_tgt_events",
    "kerberos_service_ticket_events",
    "credential_validation_events",
    "is_computer",
    "total_auth_attempts",
    "login_failure_rate"
  ],
  "thresholds": {
    "top_1pct": 0.9701639429741769,
    "top_5pct": 0.9131201006750946,
    "top_10pct": 0.8832567992038245
  },
  "limitations": [
    "Heuristic labels are not

In [ ]:
import joblib
from pathlib import Path

model_path = Path(
    "/content/drive/MyDrive/CyberDNA_LANL/"
    "cyberdna_v2_experiment/"
    "cyberdna_v2_behavioral_baseline.joblib"
)

saved = joblib.load(model_path)

print("=== Saved dictionary keys ===")
print(list(saved.keys()))

print("\n=== Contents ===")
for key, value in saved.items():
    print(f"\nKey: {key}")
    print("Type:", type(value).__name__)

    if hasattr(value, "get_params"):
        print("Parameters:", value.get_params())

    elif hasattr(value, "feature_names_in_"):
        print("Input features:", list(value.feature_names_in_))

    elif isinstance(value, (list, tuple)):
        print("Value:", list(value)[:100])

    elif isinstance(value, dict):
        print("Subkeys:", list(value.keys()))

    else:
        print("Value:", str(value)[:500])

=== Saved dictionary keys ===
['model', 'scaler', 'features', 'user_baselines', 'thresholds', 'weights', 'training_split', 'fit_population']

=== Contents ===

Key: model
Type: IsolationForest
Parameters: {'bootstrap': False, 'contamination': 'auto', 'max_features': 1.0, 'max_samples': 'auto', 'n_estimators': 200, 'n_jobs': -1, 'random_state': 2026, 'verbose': 0, 'warm_start': False}

Key: scaler
Type: RobustScaler
Parameters: {'copy': True, 'quantile_range': (25.0, 75.0), 'unit_variance': False, 'with_centering': True, 'with_scaling': True}

Key: features
Type: list
Value: ['successful_logons', 'logoffs', 'explicit_credential_events', 'privilege_events', 'process_starts', 'process_ends', 'kerberos_tgt_events', 'kerberos_service_ticket_events', 'credential_validation_events', 'is_computer', 'total_auth_attempts', 'login_failure_rate']

Key: user_baselines
Type: dict
Subkeys: ['ActiveDirectory$', 'Administrator', 'Anonymous', 'AppService', 'Comp000082$', 'Comp000116$', 'Comp000566$', 'C

In [ ]:
saved = joblib.load(
    "/content/drive/MyDrive/CyberDNA_LANL/"
    "cyberdna_v2_experiment/"
    "cyberdna_v2_behavioral_baseline.joblib"
)

print("=== Score weights ===")
print(saved["weights"])

print("\n=== Thresholds ===")
print(saved["thresholds"])

print("\n=== User baseline sample ===")
for user, baseline in list(saved["user_baselines"].items())[:3]:
    print(f"\nUser: {user}")
    print("Type:", type(baseline).__name__)
    print("Value:", str(baseline)[:500])

=== Score weights ===
{'global_anomaly': 0.6, 'user_deviation': 0.4}

=== Thresholds ===
{'top_1pct': 0.9701639429741769, 'top_5pct': 0.9131201006750946, 'top_10pct': 0.8832567992038245}

=== User baseline sample ===

User: ActiveDirectory$
Type: tuple
Value: (array([9.20119858, 9.20069408, 0.        , 9.02135658, 4.69134808,
       0.        , 0.        , 1.09861231, 0.        , 0.69314718,
       9.20119858, 0.        ]), array([6.44637525e-01, 6.45094216e-01, 9.99999997e-07, 7.32024670e-01,
       1.07451797e+00, 9.99999997e-07, 9.99999997e-07, 1.62880254e+00,
       9.99999997e-07, 9.99999997e-07, 6.44927382e-01, 9.99999997e-07]), 19)

User: Administrator
Type: tuple
Value: (array([0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 4.46590805, 0.        ,
       1.94591022, 0.69314718]), array([9.99999997e-07, 9.99999997e-07, 9.99999997e-07, 9.99999997e-07,
       9.99999997e-07, 9.99999997e-07, 9.99999997e-07, 9.99999997e-07,
   

In [ ]:
print("Saved artifact keys:", list(saved.keys()))

for key, value in saved.items():
    print(f"{key}: {type(value).__name__}")

print("\n=== Scaler ===")
scaler = saved.get("scaler")
if scaler is not None:
    print("Features:", getattr(scaler, "n_features_in_", "unknown"))
    print("Feature names:", getattr(scaler, "feature_names_in_", "not stored"))

print("\n=== Isolation Forest ===")
model = saved.get("model")
if model is not None:
    print("Features:", getattr(model, "n_features_in_", "unknown"))
    print("Feature names:", getattr(model, "feature_names_in_", "not stored"))

Saved artifact keys: ['model', 'scaler', 'features', 'user_baselines', 'thresholds', 'weights', 'training_split', 'fit_population']
model: IsolationForest
scaler: RobustScaler
features: list
user_baselines: dict
thresholds: dict
weights: dict
training_split: str
fit_population: str

=== Scaler ===
Features: 12
Feature names: ['successful_logons' 'logoffs' 'explicit_credential_events'
 'privilege_events' 'process_starts' 'process_ends' 'kerberos_tgt_events'
 'kerberos_service_ticket_events' 'credential_validation_events'
 'is_computer' 'total_auth_attempts' 'login_failure_rate']

=== Isolation Forest ===
Features: 12
Feature names: not stored


In [ ]:
import numpy as np
import pandas as pd

# Separate working copy
v21 = df.copy()

# The real count column; remove the broken fallback
v21 = v21.drop(columns=["failed_logins"], errors="ignore")

FEATURES_V21 = [
    "successful_logons",
    "failed_logons",
    "logoffs",
    "explicit_credential_events",
    "privilege_events",
    "process_starts",
    "process_ends",
    "kerberos_tgt_events",
    "kerberos_service_ticket_events",
    "credential_validation_events",
    "is_computer",
    "total_auth_attempts",
    "login_failure_rate",
]

missing = [c for c in FEATURES_V21 if c not in v21.columns]
if missing:
    raise ValueError(f"Missing required features: {missing}")

# Validate feature data
X = v21[FEATURES_V21].apply(pd.to_numeric, errors="raise")

if X.isna().any().any():
    raise ValueError("Missing values found in model features.")

if not np.isfinite(X.to_numpy()).all():
    raise ValueError("Infinite values found in model features.")

count_features = [c for c in FEATURES_V21 if c not in [
    "is_computer", "login_failure_rate"
]]

if (X[count_features] < 0).any().any():
    raise ValueError("Negative values found in count features.")

# Log-transform counts to reduce extreme-value skew
X_transformed = X.copy()
X_transformed[count_features] = np.log1p(X[count_features])

print("Rows:", len(X_transformed))
print("Feature count:", len(FEATURES_V21))
print("Contains failed_logons:", "failed_logons" in X_transformed.columns)
print("Train rows:", int((v21["hour_window"] < 10).sum()))
print("Test rows:", int((v21["hour_window"] >= 10).sum()))

print("\n=== Failure feature before and after transformation ===")
print(pd.DataFrame({
    "raw": X["failed_logons"].describe(),
    "log1p": X_transformed["failed_logons"].describe()
}))

print("\n=== Feature correlation with failed_logons ===")
print(
    X_transformed.corr(numeric_only=True)["failed_logons"]
    .sort_values(ascending=False)
)

Rows: 541209
Feature count: 13
Contains failed_logons: True
Train rows: 399249
Test rows: 141960

=== Failure feature before and after transformation ===
                 raw          log1p
count  541209.000000  541209.000000
mean        0.342243       0.014211
std        33.486277       0.190796
min         0.000000       0.000000
25%         0.000000       0.000000
50%         0.000000       0.000000
75%         0.000000       0.000000
max      6458.000000       8.773230

=== Feature correlation with failed_logons ===
failed_logons                     1.000000
login_failure_rate                0.809658
credential_validation_events      0.199427
total_auth_attempts               0.094554
explicit_credential_events        0.051882
privilege_events                  0.010511
process_ends                     -0.000825
process_starts                   -0.011990
logoffs                          -0.014500
kerberos_tgt_events              -0.023169
successful_logons                -0.024784
k

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score

ROOT = Path("/content/drive/MyDrive/CyberDNA_LANL")
OUT = ROOT / "cyberdna_v21_experiment"
OUT.mkdir(parents=True, exist_ok=True)

FEATURES = FEATURES_V21
COUNT_FEATURES = [
    c for c in FEATURES
    if c not in ["is_computer", "login_failure_rate"]
]

# Strict temporal split
train_mask = v21["hour_window"] < 10
test_mask = v21["hour_window"] >= 10

# Use heuristic labels only to select the normal training population.
# Do not use labels as model inputs.
train = v21.loc[train_mask].copy()
test = v21.loc[test_mask].copy()

normal_train_mask = train["true_rule_label"].eq("normal_activity")
normal_train = train.loc[normal_train_mask].copy()

if len(normal_train) < 100:
    raise ValueError("Too few normal training rows.")

# Transform count features
def transform_features(frame):
    X = frame[FEATURES].astype(float).copy()
    X[COUNT_FEATURES] = np.log1p(X[COUNT_FEATURES])
    return X

X_train = transform_features(normal_train)
X_test = transform_features(test)

if not np.isfinite(X_train.to_numpy()).all():
    raise ValueError("Invalid values in training features.")
if not np.isfinite(X_test.to_numpy()).all():
    raise ValueError("Invalid values in test features.")

# Fit scaler and Isolation Forest only on training-normal rows
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = IsolationForest(
    n_estimators=200,
    contamination="auto",
    random_state=2026,
    n_jobs=-1
)
model.fit(X_train_scaled)

# Higher score means more anomalous
train_global = -model.score_samples(X_train_scaled)
test_global = -model.score_samples(X_test_scaled)

# User baselines: fit on normal training data only.
# Require at least 20 rows for a stable user-specific baseline.
user_baselines = {}
for user, group in X_train.groupby(normal_train["user"]):
    if len(group) < 20:
        continue
    arr = group.to_numpy()
    user_baselines[user] = (
        arr.mean(axis=0),
        arr.std(axis=0) + 1e-6,
        len(arr)
    )

# Calculate per-row deviation from each user's baseline.
# Users without a reliable baseline receive no user-specific boost.
user_deviation = np.zeros(len(test), dtype=float)
test_users = test["user"].astype(str).to_numpy()
test_matrix = X_test.to_numpy()

for i, user in enumerate(test_users):
    baseline = user_baselines.get(user)
    if baseline is None:
        continue

    mean, std, count = baseline
    z = np.abs((test_matrix[i] - mean) / std)
    user_deviation[i] = np.mean(np.minimum(z, 10.0))

def percentile_rank(reference, values):
    reference = np.sort(np.asarray(reference))
    return np.searchsorted(reference, values, side="right") / len(reference)

# Calibrate both components against training-normal scores.
train_deviation = np.zeros(len(X_train), dtype=float)
for i, user in enumerate(normal_train["user"].astype(str).to_numpy()):
    baseline = user_baselines.get(user)
    if baseline is None:
        continue
    mean, std, _ = baseline
    z = np.abs((X_train.to_numpy()[i] - mean) / std)
    train_deviation[i] = np.mean(np.minimum(z, 10.0))

global_pct = percentile_rank(train_global, test_global)
deviation_pct = percentile_rank(train_deviation, user_deviation)

# Keep the original 60/40 weighting for a fairer comparison.
combined_score = 0.6 * global_pct + 0.4 * deviation_pct

results = test[[
    "user", "host", "hour_window", "failed_logons",
    "true_rule_label", "audit_label"
]].copy()

results["global_anomaly_percentile"] = global_pct
results["user_deviation_percentile"] = deviation_pct
results["combined_score"] = combined_score

# Thresholds are based on the training-normal combined scores.
train_global_pct = percentile_rank(train_global, train_global)
train_deviation_pct = percentile_rank(train_deviation, train_deviation)
train_combined = 0.6 * train_global_pct + 0.4 * train_deviation_pct

thresholds = {
    "top_1pct": float(np.quantile(train_combined, 0.99)),
    "top_5pct": float(np.quantile(train_combined, 0.95)),
    "top_10pct": float(np.quantile(train_combined, 0.90))
}

for name, threshold in thresholds.items():
    results[f"alert_{name}"] = results["combined_score"] >= threshold

results.sort_values("combined_score", ascending=False).to_csv(
    OUT / "test

SyntaxError: unterminated string literal (detected at line 139) (4257088021.py, line 139)

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score

ROOT = Path("/content/drive/MyDrive/CyberDNA_LANL")
OUT = ROOT / "cyberdna_v21_experiment"
OUT.mkdir(parents=True, exist_ok=True)

FEATURES = FEATURES_V21
COUNT_FEATURES = [
    c for c in FEATURES
    if c not in ["is_computer", "login_failure_rate"]
]

# Strict temporal split
train_mask = v21["hour_window"] < 10
test_mask = v21["hour_window"] >= 10

# Use heuristic labels only to select the normal training population.
# Do not use labels as model inputs.
train = v21.loc[train_mask].copy()
test = v21.loc[test_mask].copy()

normal_train_mask = train["true_rule_label"].eq("normal_activity")
normal_train = train.loc[normal_train_mask].copy()

if len(normal_train) < 100:
    raise ValueError("Too few normal training rows.")

# Transform count features
def transform_features(frame):
    X = frame[FEATURES].astype(float).copy()
    X[COUNT_FEATURES] = np.log1p(X[COUNT_FEATURES])
    return X

X_train = transform_features(normal_train)
X_test = transform_features(test)

if not np.isfinite(X_train.to_numpy()).all():
    raise ValueError("Invalid values in training features.")
if not np.isfinite(X_test.to_numpy()).all():
    raise ValueError("Invalid values in test features.")

# Fit scaler and Isolation Forest only on training-normal rows
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = IsolationForest(
    n_estimators=200,
    contamination="auto",
    random_state=2026,
    n_jobs=-1
)
model.fit(X_train_scaled)

# Higher score means more anomalous
train_global = -model.score_samples(X_train_scaled)
test_global = -model.score_samples(X_test_scaled)

# User baselines: fit on normal training data only.
# Require at least 20 rows for a stable user-specific baseline.
user_baselines = {}
for user, group in X_train.groupby(normal_train["user"]):
    if len(group) < 20:
        continue
    arr = group.to_numpy()
    user_baselines[user] = (
        arr.mean(axis=0),
        arr.std(axis=0) + 1e-6,
        len(arr)
    )

# Calculate per-row deviation from each user's baseline.
# Users without a reliable baseline receive no user-specific boost.
user_deviation = np.zeros(len(test), dtype=float)
test_users = test["user"].astype(str).to_numpy()
test_matrix = X_test.to_numpy()

for i, user in enumerate(test_users):
    baseline = user_baselines.get(user)
    if baseline is None:
        continue

    mean, std, count = baseline
    z = np.abs((test_matrix[i] - mean) / std)
    user_deviation[i] = np.mean(np.minimum(z, 10.0))

def percentile_rank(reference, values):
    reference = np.sort(np.asarray(reference))
    return np.searchsorted(reference, values, side="right") / len(reference)

# Calibrate both components against training-normal scores.
train_deviation = np.zeros(len(X_train), dtype=float)
for i, user in enumerate(normal_train["user"].astype(str).to_numpy()):
    baseline = user_baselines.get(user)
    if baseline is None:
        continue
    mean, std, _ = baseline
    z = np.abs((X_train.to_numpy()[i] - mean) / std)
    train_deviation[i] = np.mean(np.minimum(z, 10.0))

global_pct = percentile_rank(train_global, test_global)
deviation_pct = percentile_rank(train_deviation, user_deviation)

# Keep the original 60/40 weighting for a fairer comparison.
combined_score = 0.6 * global_pct + 0.4 * deviation_pct

results = test[[
    "user", "host", "hour_window", "failed_logons",
    "true_rule_label", "audit_label"
]].copy()

results["global_anomaly_percentile"] = global_pct
results["user_deviation_percentile"] = deviation_pct
results["combined_score"] = combined_score

# Thresholds are based on the training-normal combined scores.
train_global_pct = percentile_rank(train_global, train_global)
train_deviation_pct = percentile_rank(train_deviation, train_deviation)
train_combined = 0.6 * train_global_pct + 0.4 * train_deviation_pct

thresholds = {
    "top_1pct": float(np.quantile(train_combined, 0.99)),
    "top_5pct": float(np.quantile(train_combined, 0.95)),
    "top_10pct": float(np.quantile(train_combined, 0.90))
}

for name, threshold in thresholds.items():
    results[f"alert_{name}"] = results["combined_score"] >= threshold

results.sort_values("combined_score", ascending=False).to_csv(
    OUT / "test_scored_profiles.csv", index=False
)

artifact = {
    "model": model,
    "scaler": scaler,
    "features": FEATURES,
    "count_features_log1p": COUNT_FEATURES,
    "user_baselines": user_baselines,
    "thresholds": thresholds,
    "weights": {"global_anomaly": 0.6, "user_deviation": 0.4},
    "training_split": "hour_window < 10",
    "fit_population": "normal_activity training rows only",
}

joblib.dump(artifact, OUT / "cyberdna_v21_behavioral_baseline.joblib")

report = {
    "experiment": "Cyber DNA v2.1 corrected feature baseline",
    "rows": {
        "training_period": int(len(train)),
        "normal_training": int(len(normal_train)),
        "test_period": int(len(test)),
        "test_high_failure_profiles": int(
            (test["true_rule_label"] == "high_failure_profile").sum()
        )
    },
    "features": FEATURES,
    "thresholds": thresholds,
    "limitations": [
        "Heuristic labels are not independent attack ground truth.",
        "Anomaly scores indicate unusual activity, not confirmed maliciousness.",
        "Thresholds are calibrated on training-normal scores.",
        "User baselines are unavailable for users with fewer than 20 normal training rows."
    ]
}

with open(OUT / "experiment_report.json", "w") as f:
    json.dump(report, f, indent=2)

print("Saved to:", OUT)
print("Features:", len(FEATURES))
print("User baselines:", len(user_baselines))
print("Thresholds:", thresholds)

print("\n=== Test alert counts ===")
for name in thresholds:
    col = f"alert_{name}"
    print(name, int(results[col].sum()))

print("\n=== Top 15 profiles ===")
print(results.nlargest(15, "combined_score")[[
    "user", "host", "hour_window", "failed_logons",
    "true_rule_label", "combined_score"
]].to_string(index=False))

print("\n=== High-failure profile ranking ===")
high = results["true_rule_label"].eq("high_failure_profile")
print("High-failure rows:", int(high.sum()))
if high.any():
    print(
        results.loc[high, "combined_score"]
        .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
    )
    for name in thresholds:
        col = f"alert_{name}"
        print(
            f"{name}: {int(results.loc[high, col].sum())}/"
            f"{int(high.sum())} high-failure profiles flagged"
        )

Saved to: /content/drive/MyDrive/CyberDNA_LANL/cyberdna_v21_experiment
Features: 13
User baselines: 11135
Thresholds: {'top_1pct': 0.9704783925848648, 'top_5pct': 0.9004287376615038, 'top_10pct': 0.8483361588155045}

=== Test alert counts ===
top_1pct 1450
top_5pct 6617
top_10pct 11526

=== Top 15 profiles ===
                user                host  hour_window  failed_logons          true_rule_label  combined_score
          AppService EnterpriseAppServer           12         1391.0 privileged_admin_profile        1.000000
          AppService EnterpriseAppServer           10         2237.0 privileged_admin_profile        1.000000
          AppService EnterpriseAppServer           11         2091.0 privileged_admin_profile        1.000000
             Scanner     ActiveDirectory           12            0.0 privileged_admin_profile        0.999989
       Administrator     ActiveDirectory           11           30.0          normal_activity        0.999966
             Scanner     Act

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/CyberDNA_LANL")
EXP = ROOT / "cyberdna_v22_experiment"

file_path = EXP / "v24_directional_change_scores.csv"

print("File exists:", file_path.exists())

if file_path.exists():
    df = pd.read_csv(file_path)
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

File exists: True
Rows: 141960
Columns: ['user', 'host', 'hour_window', 'successful_logons', 'failed_logons', 'logoffs', 'explicit_credential_events', 'privilege_events', 'process_starts', 'process_ends', 'kerberos_tgt_events', 'kerberos_service_ticket_events', 'credential_validation_events', 'previous_hour', 'log_successful_logons', 'signed_successful_logons', 'log_failed_logons', 'signed_failed_logons', 'log_privilege_events', 'signed_privilege_events', 'log_process_starts', 'signed_process_starts', 'log_explicit_credential_events', 'signed_explicit_credential_events', 'log_logoffs', 'signed_logoffs', 'log_kerberos_tgt_events', 'signed_kerberos_tgt_events', 'log_kerberos_service_ticket_events', 'signed_kerberos_service_ticket_events', 'log_credential_validation_events', 'signed_credential_validation_events', 'abs_successful_logons', 'abs_failed_logons', 'abs_privilege_events', 'abs_process_starts', 'abs_explicit_credential_events', 'abs_logoffs', 'abs_kerberos_tgt_events', 'abs_kerbe

In [ ]:

import json
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.ensemble import IsolationForest

# ============================================================
# 1. CONFIGURATION
# ============================================================

ROOT = Path("/content/drive/MyDrive/CyberDNA_LANL")
OUT = ROOT / "v25_final"
OUT.mkdir(parents=True, exist_ok=True)

DATA = ROOT / "lanl_host_aware_aggregated.parquet"

features = [
    "successful_logons",
    "failed_logons",
    "logoffs",
    "explicit_credential_events",
    "privilege_events",
    "process_starts",
    "kerberos_tgt_events",
    "kerberos_service_ticket_events",
    "credential_validation_events",
    "explicit_credential_events",
]

# Remove accidental duplicate feature names.
features = list(dict.fromkeys(features))

# process_ends is intentionally excluded pending data validation.
df = pd.read_parquet(DATA)

required = ["user", "host", "hour_window"] + features
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df[required].copy()
df["hour_window"] = pd.to_numeric(
    df["hour_window"], errors="raise"
)

for f in features:
    df[f] = pd.to_numeric(df[f], errors="raise").fillna(0)
    if (df[f] < 0).any():
        raise ValueError(f"Negative counts found in {f}")

df = df.sort_values(
    ["user", "host", "hour_window"]
).reset_index(drop=True)

print("Dataset:", df.shape)
print("Features:", features)

# ============================================================
# 2. BUILD CONSECUTIVE-HOUR CHANGES
# ============================================================

grouped = df.groupby(["user", "host"], sort=False)

previous_hour = grouped["hour_window"].shift(1)
gap = df["hour_window"] - previous_hour

delta = pd.DataFrame(index=df.index)

for f in features:
    previous = grouped[f].shift(1)
    delta[f] = df[f] - previous

# A row is usable only if the same user-host exists
# in the immediately preceding hour.
valid = gap.eq(1)

# Train: hours 1-7
# Calibration: hours 8-9
# Test: hours 10-12
train_mask = valid & df["hour_window"].between(1, 7)
cal_mask = valid & df["hour_window"].between(8, 9)
test_mask = valid & df["hour_window"].between(10, 12)

if train_mask.sum() < 100 or cal_mask.sum() < 100:
    raise ValueError("Too few training or calibration rows.")

# Signed log changes preserve increase vs decrease.
signed_change = pd.DataFrame(index=df.index)

for f in features:
    x = delta[f].to_numpy(dtype=float)
    signed_change[f] = np.sign(x) * np.log1p(np.abs(x))

# Fit feature scales using training data only.
scales = {}

for f in features:
    scale = signed_change.loc[train_mask, f].abs().quantile(0.99)
    scales[f] = max(float(scale), 0.10)

X = signed_change.copy()

for f in features:
    X[f] = (X[f] / scales[f]).clip(-5, 5)

X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

# ============================================================
# 3. TRAIN THE MODEL
# ============================================================

model = IsolationForest(
    n_estimators=300,
    max_samples="auto",
    contamination="auto",
    random_state=2026,
    n_jobs=-1,
)

model.fit(X.loc[train_mask, features])

# Higher score = more anomalous.
anomaly_score = pd.Series(
    np.nan, index=df.index, dtype=float
)

anomaly_score.loc[valid] = -model.decision_function(
    X.loc[valid, features]
)

# ============================================================
# 4. CALIBRATE THRESHOLDS ON HOURS 8-9
# ============================================================

cal_scores = anomaly_score.loc[cal_mask].dropna()

threshold_model_99 = float(cal_scores.quantile(0.99))
threshold_model_995 = float(cal_scores.quantile(0.995))

# Directional scores: positive changes vs negative changes.
increase_score = pd.Series(0.0, index=df.index)
decrease_score = pd.Series(0.0, index=df.index)

for f in features:
    z = X[f]
    increase_score += z.clip(lower=0)
    decrease_score += (-z).clip(lower=0)

increase_score /= len(features)
decrease_score /= len(features)

threshold_increase_995 = float(
    increase_score.loc[cal_mask].quantile(0.995)
)
threshold_decrease_995 = float(
    decrease_score.loc[cal_mask].quantile(0.995)
)

thresholds = {
    "model_score_p99": threshold_model_99,
    "model_score_p995": threshold_model_995,
    "increase_score_p995": threshold_increase_995,
    "decrease_score_p995": threshold_decrease_995,
    "calibration_hours": [8, 9],
    "training_hours": [1, 2, 3, 4, 5, 6, 7],
    "test_hours": [10, 11, 12],
    "feature_names": features,
    "excluded_features": {
        "process_ends": "Excluded pending data-quality validation"
    },
    "score_meaning": (
        "Anomaly score, not attack probability or proof of compromise"
    ),
}

# ============================================================
# 5. SAVE ARTIFACTS
# ============================================================

joblib.dump(model, OUT / "cyberdna_v25_isolation_forest.joblib")

with open(OUT / "cyberdna_v25_config.json", "w") as f:
    json.dump(
        {
            "features": features,
            "scales": scales,
            "thresholds": thresholds,
            "random_state": 2026,
            "n_estimators": 300,
        },
        f,
        indent=2,
    )

results = df[
    ["user", "host", "hour_window"]
].copy()

results["anomaly_score"] = anomaly_score
results["increase_score"] = increase_score
results["decrease_score"] = decrease_score
results["split"] = np.select(
    [train_mask, cal_mask, test_mask],
    ["train", "calibration", "test"],
    default="unscored",
)

results["model_candidate_p99"] = (
    test_mask & (anomaly_score >= threshold_model_99)
)
results["model_candidate_p995"] = (
    test_mask & (anomaly_score >= threshold_model_995)
)
results["increase_candidate_p995"] = (
    test_mask & (increase_score >= threshold_increase_995)
)
results["decrease_candidate_p995"] = (
    test_mask & (decrease_score >= threshold_decrease_995)
)

results.to_csv(
    OUT / "cyberdna_v25_scores.csv",
    index=False,
)

with open(OUT / "cyberdna_v25_thresholds.json", "w") as f:
    json.dump(thresholds, f, indent=2)

# ============================================================
# 6. REPORT TEST RESULTS
# ============================================================

test_results = results.loc[test_mask].copy()
n_test = len(test_results)

print("\n===== CYBER DNA V25 TRAINING COMPLETE =====")
print("Training rows:", int(train_mask.sum()))
print("Calibration rows:", int(cal_mask.sum()))
print("Test rows:", n_test)
print("Skipped non-consecutive rows:", int((~valid).sum()))

print("\nCalibration thresholds:")
print(json.dumps(thresholds, indent=2))

print("\nTest candidate counts:")
for column in [
    "model_candidate_p99",
    "model_candidate_p995",
    "increase_candidate_p995",
    "decrease_candidate_p995",
]:
    count = int(test_results[column].sum())
    rate = 100 * count / max(n_test, 1)
    print(f"{column}: {count} ({rate:.3f}%)")

print("\nHighest-scoring test candidates:")
display(
    test_results.sort_values(
        "anomaly_score", ascending=False
    ).head(20)
)

print("\nSaved files:")
for p in sorted(OUT.iterdir()):
    print(p.name, f"({p.stat().st_size:,} bytes)")

print(
    "\nImportant: candidate rates are not accuracy, "
    "precision, recall, or false-positive rates."
)


In [ ]:

import json
import joblib
from pathlib import Path

OUT = Path("/content/drive/MyDrive/CyberDNA_LANL/v25_final")

model = joblib.load(
    OUT / "cyberdna_v25_isolation_forest.joblib"
)

with open(OUT / "cyberdna_v25_config.json") as f:
    config = json.load(f)

print("Model loaded:", type(model).__name__)
print("Feature count:", len(config["features"]))
print("Features:", config["features"])
print("Thresholds:", config["thresholds"])
print("Verification: PASS")


Model loaded: IsolationForest
Feature count: 9
Features: ['successful_logons', 'failed_logons', 'logoffs', 'explicit_credential_events', 'privilege_events', 'process_starts', 'kerberos_tgt_events', 'kerberos_service_ticket_events', 'credential_validation_events']
Thresholds: {'model_score_p99': 0.10532385168133324, 'model_score_p995': 0.1226613500301712, 'increase_score_p995': 0.6601691370620053, 'decrease_score_p995': 0.6705142140817397, 'calibration_hours': [8, 9], 'training_hours': [1, 2, 3, 4, 5, 6, 7], 'test_hours': [10, 11, 12], 'feature_names': ['successful_logons', 'failed_logons', 'logoffs', 'explicit_credential_events', 'privilege_events', 'process_starts', 'kerberos_tgt_events', 'kerberos_service_ticket_events', 'credential_validation_events'], 'excluded_features': {'process_ends': 'Excluded pending data-quality validation'}, 'score_meaning': 'Anomaly score, not attack probability or proof of compromise'}
Verification: PASS


In [ ]:

# Cell 1 — Connect Google Drive
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

!apt-get -qq install -y git-lfs
!git lfs install

from pathlib import Path

ROOT = Path("/content/drive/MyDrive/CyberDNA_LANL")
ROOT.mkdir(parents=True, exist_ok=True)

REPO = ROOT / "splunk_attack_data"

if not (REPO / ".git").exists():
    !git clone --depth 1 https://github.com/splunk/attack_data.git "{REPO}"
else:
    print("Repository already exists:", REPO)

print("Repository path:", REPO)


Selecting previously unselected package git-lfs.
(Reading database ... 122807 files and directories currently installed.)
Preparing to unpack .../git-lfs_3.4.1-1ubuntu0.4_amd64.deb ...
Unpacking git-lfs (3.4.1-1ubuntu0.4) ...
Setting up git-lfs (3.4.1-1ubuntu0.4) ...
Processing triggers for man-db (2.12.0-4build2) ...
Git LFS initialized.
Cloning into '/content/drive/MyDrive/CyberDNA_LANL/splunk_attack_data'...
remote: Enumerating objects: 5568, done.
remote: Counting objects: 100% (5568/5568), done.
remote: Compressing objects: 100% (4798/4798), done.
remote: Total 5568 (delta 762), reused 4238 (delta 626), pack-reused 0 (from 0)
Receiving objects: 100% (5568/5568), 15.59 MiB | 15.53 MiB/s, done.
Resolving deltas: 100% (762/762), done.
Updating files: 100% (3787/3787), done.
Filtering content: 100% (2304/2304), 11.31 GiB | 23.83 MiB/s, done.
fatal: cannot exec '/content/drive/MyDrive/CyberDNA_LANL/splunk_attack_data/.git/hooks/post-checkout': Permission denied
Repository path: /conten

In [ ]:

REPO = "/content/drive/MyDrive/CyberDNA_LANL/splunk_attack_data"

!find "{REPO}/datasets/attack_techniques" -maxdepth 1 -mindepth 1 -type d | head -30


/content/drive/MyDrive/CyberDNA_LANL/splunk_attack_data/datasets/attack_techniques/T1001
/content/drive/MyDrive/CyberDNA_LANL/splunk_attack_data/datasets/attack_techniques/T1003.001
/content/drive/MyDrive/CyberDNA_LANL/splunk_attack_data/datasets/attack_techniques/T1003.002
/content/drive/MyDrive/CyberDNA_LANL/splunk_attack_data/datasets/attack_techniques/T1003.003
/content/drive/MyDrive/CyberDNA_LANL/splunk_attack_data/datasets/attack_techniques/T1003.004
/content/drive/MyDrive/CyberDNA_LANL/splunk_attack_data/datasets/attack_techniques/T1003.005
/content/drive/MyDrive/CyberDNA_LANL/splunk_attack_data/datasets/attack_techniques/T1003.006
/content/drive/MyDrive/CyberDNA_LANL/splunk_attack_data/datasets/attack_techniques/T1003.008
/content/drive/MyDrive/CyberDNA_LANL/splunk_attack_data/datasets/attack_techniques/T1003
/content/drive/MyDrive/CyberDNA_LANL/splunk_attack_data/datasets/attack_techniques/T1005
/content/drive/MyDrive/CyberDNA_LANL/splunk_attack_data/datasets/attack_techniques

In [ ]:

from pathlib import Path
import shutil

paths = [
    Path("/content/drive/MyDrive/CyberDNA_LANL"),
    Path("/content/drive/MyDrive/CyberDNA_LANL/splunk_attack_data"),
]

for path in paths:
    if path.exists():
        size_gb = sum(
            f.stat().st_size
            for f in path.rglob("*")
            if f.is_file()
        ) / (1024 ** 3)
        print(f"{path}: {size_gb:.2f} GB")

total, used, free = shutil.disk_usage("/content")
print(f"\nColab temporary disk free: {free / (1024**3):.2f} GB")


/content/drive/MyDrive/CyberDNA_LANL: 23.34 GB
/content/drive/MyDrive/CyberDNA_LANL/splunk_attack_data: 22.62 GB

Colab temporary disk free: 64.26 GB


In [ ]:

from pathlib import Path

target = Path(
    "/content/drive/MyDrive/CyberDNA_LANL/splunk_attack_data"
)

if target.exists():
    total = sum(
        f.stat().st_size
        for f in target.rglob("*")
        if f.is_file()
    )
    print("Folder:", target)
    print(f"Size: {total / (1024**3):.2f} GB")
    print("Exists:", target.exists())
    print("\nThis is the downloaded Splunk repository only.")
else:
    print("Folder not found. Nothing to delete.")


Folder: /content/drive/MyDrive/CyberDNA_LANL/splunk_attack_data
Size: 22.62 GB
Exists: True

This is the downloaded Splunk repository only.


In [ ]:

from pathlib import Path
import shutil

target = Path(
    "/content/drive/MyDrive/CyberDNA_LANL/splunk_attack_data"
)

if not target.exists():
    print("Folder not found. Nothing to delete.")
else:
    confirm = input(
        "Delete the 22.62 GB Splunk repository? Type YES to confirm: "
    )

    if confirm == "YES":
        shutil.rmtree(target)
        print("Splunk repository deleted.")
        print("Your LANL dataset and V25 model files were not targeted.")
    else:
        print("Cancelled. Nothing was deleted.")


Delete the 22.62 GB Splunk repository? Type YES to confirm: yes
Cancelled. Nothing was deleted.


In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/CyberDNA_LANL")

for p in sorted(ROOT.iterdir()):
    kind = "Folder" if p.is_dir() else "File"
    print(f"{kind}: {p.name}")

Folder: cyberdna_v21_experiment
Folder: cyberdna_v22_experiment
Folder: cyberdna_v2_experiment
Folder: cyberdna_validation_harness
Folder: host_aware_partials
File: lanl_features.parquet
File: lanl_host_aware_aggregated.parquet
Folder: partials
Folder: production_v1
Folder: splunk_attack_data
Folder: v25_final
File: wls_day-01.bz2


In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/CyberDNA_LANL")

items = []

for p in ROOT.iterdir():
    size = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) if p.is_dir() else p.stat().st_size
    items.append((size / (1024**3), p.name, "Folder" if p.is_dir() else "File"))

for size, name, kind in sorted(items, reverse=True):
    print(f"{size:8.2f} GB | {kind:6} | {name}")

    0.38 GB | File   | wls_day-01.bz2
    0.12 GB | Folder | host_aware_partials
    0.09 GB | Folder | cyberdna_v22_experiment
    0.05 GB | Folder | v25_final
    0.05 GB | Folder | partials
    0.02 GB | Folder | cyberdna_v21_experiment
    0.01 GB | Folder | cyberdna_v2_experiment
    0.00 GB | File   | lanl_host_aware_aggregated.parquet
    0.00 GB | File   | lanl_features.parquet
    0.00 GB | Folder | production_v1
    0.00 GB | Folder | cyberdna_validation_harness


In [ ]:
from pathlib import Path
import json
import joblib

MODEL_DIR = Path(
    "/content/drive/MyDrive/CyberDNA_LANL/v25_final"
)

required_files = [
    "cyberdna_v25_isolation_forest.joblib",
    "cyberdna_v25_config.json",
    "cyberdna_v25_thresholds.json",
    "cyberdna_v25_scores.csv",
]

for name in required_files:
    path = MODEL_DIR / name
    print(f"{'OK' if path.exists() else 'MISSING'}: {name}")

model_path = MODEL_DIR / "cyberdna_v25_isolation_forest.joblib"

if model_path.exists():
    model = joblib.load(model_path)
    print("\nModel type:", type(model).__name__)
    print("Feature count:", model.n_features_in_)

config_path = MODEL_DIR / "cyberdna_v25_config.json"
if config_path.exists():
    print("\nConfiguration:")
    print(json.dumps(json.loads(config_path.read_text()), indent=2))

OK: cyberdna_v25_isolation_forest.joblib
OK: cyberdna_v25_config.json
OK: cyberdna_v25_thresholds.json
OK: cyberdna_v25_scores.csv

Model type: IsolationForest
Feature count: 9

Configuration:
{
  "features": [
    "successful_logons",
    "failed_logons",
    "logoffs",
    "explicit_credential_events",
    "privilege_events",
    "process_starts",
    "kerberos_tgt_events",
    "kerberos_service_ticket_events",
    "credential_validation_events"
  ],
  "scales": {
    "successful_logons": 3.4339872044851463,
    "failed_logons": 0.1,
    "logoffs": 3.4339872044851463,
    "explicit_credential_events": 1.0986122886681096,
    "privilege_events": 1.0986122886681096,
    "process_starts": 3.367295829986474,
    "kerberos_tgt_events": 1.6094379124341003,
    "kerberos_service_ticket_events": 2.639057329615259,
    "credential_validation_events": 1.9459101490553132
  },
  "thresholds": {
    "model_score_p99": 0.10532385168133324,
    "model_score_p995": 0.1226613500301712,
    "increase_

In [ ]:
%cd /content/attack_data
!git lfs pull --include="datasets/attack_techniques/T1003.001/atomic_red_team/windows-sysmon.log"

/content/attack_data


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import json

BASE = Path("/content/drive/MyDrive/CyberDNA_LANL")
DATA_FILE = BASE / "lanl_host_aware_aggregated.parquet"
V25_DIR = BASE / "v25_final"
V26_DIR = BASE / "v26_candidate"

# Check that the files exist
for path in [DATA_FILE, V25_DIR, V26_DIR]:
    print(f"{path}: {'FOUND' if path.exists() else 'MISSING'}")

if not DATA_FILE.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_FILE}")

df = pd.read_parquet(DATA_FILE)

print("\nDataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
display(df.isna().sum().to_frame("missing_count"))

print("\nDuplicate rows:", df.duplicated().sum())


/content/drive/MyDrive/CyberDNA_LANL/lanl_host_aware_aggregated.parquet: MISSING
/content/drive/MyDrive/CyberDNA_LANL/v25_final: MISSING
/content/drive/MyDrive/CyberDNA_LANL/v26_candidate: MISSING


FileNotFoundError: Dataset not found: /content/drive/MyDrive/CyberDNA_LANL/lanl_host_aware_aggregated.parquet

In [ ]:

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path

ROOT = Path("/content/drive/MyDrive")

targets = {
    "lanl_host_aware_aggregated.parquet",
    "lanl_features.parquet",
    "cyberdna_v25_isolation_forest.joblib",
    "cyberdna_v26_isolation_forest.joblib",
    "cyberdna_v25_config.json",
    "cyberdna_v26_metadata.json",
}

print("Searching Google Drive...")
found = 0

for path in ROOT.rglob("*"):
    if path.name in targets:
        print("FOUND:", path)
        found += 1

print(f"\nSearch complete. Matching files: {found}")


Mounted at /content/drive
Searching Google Drive...
FOUND: /content/drive/MyDrive/CyberDNA_LANL/lanl_features.parquet
FOUND: /content/drive/MyDrive/CyberDNA_LANL/lanl_host_aware_aggregated.parquet
FOUND: /content/drive/MyDrive/CyberDNA_LANL/v25_final/cyberdna_v25_isolation_forest.joblib
FOUND: /content/drive/MyDrive/CyberDNA_LANL/v25_final/cyberdna_v25_config.json

Search complete. Matching files: 4


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("/content/drive/MyDrive/CyberDNA_LANL")
DATA_FILE = BASE / "lanl_host_aware_aggregated.parquet"

df = pd.read_parquet(DATA_FILE)

FEATURES = [
    "successful_logons",
    "failed_logons",
    "logoffs",
    "explicit_credential_events",
    "privilege_events",
    "process_starts",
    "kerberos_tgt_events",
    "kerberos_service_ticket_events",
    "credential_validation_events",
]

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

missing = [f for f in FEATURES if f not in df.columns]
print("\nMissing model features:", missing)

if not missing:
    X = df[FEATURES]

    print("\nMissing values:", int(X.isna().sum().sum()))
    print("Infinite values:", int(np.isinf(X.to_numpy(dtype=float)).sum()))
    print("Negative values:", int((X < 0).sum().sum()))
    print("Duplicate full rows:", int(df.duplicated().sum()))

    print("\nFeature statistics:")
    display(X.describe(
        percentiles=[0.50, 0.90, 0.95, 0.99, 0.999]
    ).T)

    print("\nFeature equality checks:")
    for a, b in [
        ("successful_logons", "privilege_events"),
        ("successful_logons", "logoffs"),
        ("successful_logons", "process_starts"),
    ]:
        print(f"{a} == {b}: {(X[a] == X[b]).mean():.2%}")

print("\nV25 files:")
for p in sorted((BASE / "v25_final").glob("*")):
    print(p.name)

print("\nV26 candidate folder exists:", (BASE / "v26_candidate").exists())


Dataset shape: (541209, 13)
Columns: ['user', 'host', 'hour_window', 'successful_logons', 'failed_logons', 'logoffs', 'explicit_credential_events', 'privilege_events', 'process_starts', 'process_ends', 'kerberos_tgt_events', 'kerberos_service_ticket_events', 'credential_validation_events']

Missing model features: []

Missing values: 0
Infinite values: 0
Negative values: 0
Duplicate full rows: 0

Feature statistics:


,count,mean,std,min,50%,90%,95%,99%,99.9%,max
successful_logons,541209.0,13.642779,397.480878,0.0,1.0,21.0,26.0,119.0,782.376,126251.0
failed_logons,541209.0,0.342243,33.486277,0.0,0.0,0.0,0.0,0.0,12.000,6458.0
logoffs,541209.0,13.346212,396.746476,0.0,1.0,21.0,26.0,119.0,721.960,127602.0
explicit_credential_events,541209.0,0.921659,72.060455,0.0,0.0,0.0,1.0,6.0,63.000,21223.0
privilege_events,541209.0,4.068478,375.300477,0.0,0.0,1.0,2.0,4.0,145.000,126270.0
process_starts,541209.0,15.196006,275.822495,0.0,0.0,39.0,52.0,100.0,789.000,50611.0
kerberos_tgt_events,541209.0,0.755499,61.110214,0.0,0.0,1.0,2.0,9.0,20.000,21256.0
kerberos_service_ticket_events,541209.0,1.926781,75.663011,0.0,0.0,4.0,8.0,19.0,42.000,21258.0
credential_validation_events,541209.0,4.861137,353.342809,0.0,0.0,3.0,4.0,40.0,211.000,69328.0



Feature equality checks:
successful_logons == privilege_events: 45.69%
successful_logons == logoffs: 79.15%
successful_logons == process_starts: 5.83%

V25 files:
cyberdna_v25_config.json
cyberdna_v25_isolation_forest.joblib
cyberdna_v25_scores.csv
cyberdna_v25_thresholds.json

V26 candidate folder exists: False


In [ ]:

import pandas as pd
import numpy as np

FEATURES = [
    "successful_logons",
    "failed_logons",
    "logoffs",
    "explicit_credential_events",
    "privilege_events",
    "process_starts",
    "kerberos_tgt_events",
    "kerberos_service_ticket_events",
    "credential_validation_events",
]

# 1. Inspect the 20 rows with the most successful logons
print("=== TOP 20 SUCCESSFUL LOGON COUNTS ===")
display(
    df.nlargest(20, "successful_logons")[
        ["user", "host", "hour_window"] + FEATURES
    ]
)

# 2. Check whether feature equality is concentrated in extreme rows
print("\n=== FEATURE EQUALITY BY LOGON VOLUME ===")

for limit, label in [
    (0, "All rows"),
    (100, "Successful logons > 100"),
    (1000, "Successful logons > 1,000"),
]:
    subset = df if limit == 0 else df[df["successful_logons"] > limit]
    if len(subset) == 0:
        continue

    print(f"\n{label}: {len(subset):,} rows")
    for a, b in [
        ("successful_logons", "privilege_events"),
        ("successful_logons", "logoffs"),
        ("successful_logons", "process_starts"),
    ]:
        print(f"  {a} == {b}: {(subset[a] == subset[b]).mean():.2%}")

# 3. Find the users and hosts with the largest total counts
print("\n=== TOP 15 USERS BY SUCCESSFUL LOGONS ===")
display(
    df.groupby("user")["successful_logons"]
      .agg(["count", "sum", "max"])
      .sort_values("sum", ascending=False)
      .head(15)
)

print("\n=== TOP 15 HOSTS BY SUCCESSFUL LOGONS ===")
display(
    df.groupby("host")["successful_logons"]
      .agg(["count", "sum", "max"])
      .sort_values("sum", ascending=False)
      .head(15)
)

# 4. Check how many windows each user-host pair has
pair_counts = df.groupby(["user", "host"]).size()

print("\n=== USER-HOST WINDOW COUNTS ===")
print(pair_counts.describe(percentiles=[0.5, 0.9, 0.99]))
print("Pairs with more than 13 rows:", int((pair_counts > 13).sum()))

# 5. Check V26 artifact availability without modifying anything
BASE = "/content/drive/MyDrive/CyberDNA_LANL"
from pathlib import Path

print("\n=== MODEL ARTIFACTS ===")
for p in [
    Path(BASE) / "v25_final" / "cyberdna_v25_isolation_forest.joblib",
    Path(BASE) / "v26_candidate" / "cyberdna_v26_isolation_forest.joblib",
    Path(BASE) / "v26_candidate" / "cyberdna_v26_metadata.json",
    Path(BASE) / "v26_candidate" / "cyberdna_v26_test_scores.csv",
]:
    print(f"{'FOUND' if p.exists() else 'MISSING'}: {p}")


=== TOP 20 SUCCESSFUL LOGON COUNTS ===


,user,host,hour_window,successful_logons,failed_logons,logoffs,explicit_credential_events,privilege_events,process_starts,kerberos_tgt_events,kerberos_service_ticket_events,credential_validation_events
389922,Comp916004$,Comp916004,9,126251.0,29.0,127602.0,390.0,126270.0,11554.0,0.0,0.0,0.0
418260,Comp916004$,Comp916004,10,114141.0,31.0,114649.0,319.0,114162.0,11386.0,0.0,0.0,0.0
118713,Comp916004$,Comp916004,8,99326.0,31.0,98451.0,258.0,99367.0,11542.0,0.0,0.0,0.0
439403,Comp916004$,Comp916004,11,90954.0,29.0,89213.0,188.0,90952.0,11407.0,0.0,0.0,0.0
100386,Comp916004$,Comp916004,7,74330.0,32.0,73271.0,155.0,74346.0,11346.0,0.0,0.0,0.0
336707,Comp916004$,Comp916004,12,66253.0,24.0,65514.0,108.0,66257.0,10322.0,0.0,0.0,0.0
83489,Comp916004$,Comp916004,6,44741.0,27.0,42883.0,52.0,44745.0,11455.0,0.0,0.0,0.0
127933,Comp916004$,Comp916004,0,39002.0,29.0,37841.0,29.0,39000.0,11453.0,0.0,0.0,0.0
192902,Comp916004$,Comp916004,5,37875.0,33.0,36478.0,14.0,37879.0,11353.0,0.0,0.0,0.0
16732,Comp916004$,Comp916004,1,37471.0,35.0,36683.0,24.0,37479.0,11479.0,0.0,0.0,0.0



=== FEATURE EQUALITY BY LOGON VOLUME ===

All rows: 541,209 rows
  successful_logons == privilege_events: 45.69%
  successful_logons == logoffs: 79.15%
  successful_logons == process_starts: 5.83%

Successful logons > 100: 6,453 rows
  successful_logons == privilege_events: 4.34%
  successful_logons == logoffs: 45.54%
  successful_logons == process_starts: 0.14%

Successful logons > 1,000: 478 rows
  successful_logons == privilege_events: 20.29%
  successful_logons == logoffs: 16.95%
  successful_logons == process_starts: 0.00%

=== TOP 15 USERS BY SUCCESSFUL LOGONS ===


,count,sum,max
user,,,
Comp916004$,43,1005182.0,126251.0
AppService,350,437519.0,31822.0
Scanner,1372,311068.0,10852.0
Administrator,1972,183748.0,12086.0
ActiveDirectory$,28,156818.0,15305.0
Comp915658$,26,133636.0,11980.0
Anonymous,3268,116314.0,5852.0
system,59859,116010.0,489.0
User793535,60,93643.0,8488.0



=== TOP 15 HOSTS BY SUCCESSFUL LOGONS ===


,count,sum,max
host,,,
ActiveDirectory,180919,3583914.0,15305.0
Comp916004,4180,954088.0,126251.0
EnterpriseAppServer,2193,510752.0,31822.0
Comp704126,2630,343304.0,718.0
Comp081330,4914,154419.0,867.0
Comp256596,1995,149515.0,795.0
Comp698972,432,94111.0,10328.0
Comp894088,410,91787.0,10332.0
Comp897367,75,89805.0,21219.0



=== USER-HOST WINDOW COUNTS ===
count    87072.000000
mean         6.215649
std          4.923547
min          1.000000
50%          4.000000
90%         13.000000
99%         13.000000
max         13.000000
dtype: float64
Pairs with more than 13 rows: 0

=== MODEL ARTIFACTS ===
FOUND: /content/drive/MyDrive/CyberDNA_LANL/v25_final/cyberdna_v25_isolation_forest.joblib
MISSING: /content/drive/MyDrive/CyberDNA_LANL/v26_candidate/cyberdna_v26_isolation_forest.joblib
MISSING: /content/drive/MyDrive/CyberDNA_LANL/v26_candidate/cyberdna_v26_metadata.json
MISSING: /content/drive/MyDrive/CyberDNA_LANL/v26_candidate/cyberdna_v26_test_scores.csv


In [ ]:

import numpy as np
import pandas as pd

FEATURES = [
    "successful_logons",
    "failed_logons",
    "logoffs",
    "explicit_credential_events",
    "privilege_events",
    "process_starts",
    "kerberos_tgt_events",
    "kerberos_service_ticket_events",
    "credential_validation_events",
]

# 1. Correlation between event counts
print("=== FEATURE CORRELATIONS ===")
display(df[FEATURES].corr().round(2))

# 2. Investigate rows where logons and privilege events are nearly equal
check = df[["user", "host", "hour_window"] + FEATURES].copy()

check["logon_privilege_difference"] = (
    check["successful_logons"] - check["privilege_events"]
).abs()

check["logon_logoff_difference"] = (
    check["successful_logons"] - check["logoffs"]
).abs()

check["high_logon_volume"] = check["successful_logons"] > 1000

print("\n=== HIGH-VOLUME ROWS WITH NEARLY EQUAL LOGON/PRIVILEGE COUNTS ===")
display(
    check[
        check["high_logon_volume"]
        & (check["logon_privilege_difference"] <= 2)
    ].sort_values("successful_logons", ascending=False).head(20)
)

# 3. Summarize extreme counts by user-host pair
pair_summary = (
    df.groupby(["user", "host"])
      .agg(
          windows=("hour_window", "nunique"),
          max_successful_logons=("successful_logons", "max"),
          max_privilege_events=("privilege_events", "max"),
          max_logoffs=("logoffs", "max"),
      )
      .sort_values("max_successful_logons", ascending=False)
)

print("\n=== TOP 20 USER-HOST PAIRS ===")
display(pair_summary.head(20))

# 4. Create a review list; these are not confirmed errors or attacks
review = df[
    (df["successful_logons"] > 1000)
    | (df["failed_logons"] > 100)
    | (df["explicit_credential_events"] > 1000)
    | (df["privilege_events"] > 1000)
].copy()

print("\nRows flagged for data-quality review:", len(review))
print("Share of dataset:", f"{len(review) / len(df):.3%}")

print("\nV27 status: diagnostics only; original dataset remains unchanged.")


=== FEATURE CORRELATIONS ===


,successful_logons,failed_logons,logoffs,explicit_credential_events,privilege_events,process_starts,kerberos_tgt_events,kerberos_service_ticket_events,credential_validation_events
successful_logons,1.00,0.07,1.00,0.13,0.95,0.17,0.04,0.11,0.11
failed_logons,0.07,1.00,0.07,0.02,0.07,0.01,-0.00,-0.00,0.04
logoffs,1.00,0.07,1.00,0.13,0.95,0.17,0.04,0.11,0.11
explicit_credential_events,0.13,0.02,0.13,1.00,0.02,0.00,-0.00,-0.00,0.03
privilege_events,0.95,0.07,0.95,0.02,1.00,0.18,0.00,0.05,0.02
process_starts,0.17,0.01,0.17,0.00,0.18,1.00,-0.00,-0.00,0.00
kerberos_tgt_events,0.04,-0.00,0.04,-0.00,0.00,-0.00,1.00,0.90,0.31
kerberos_service_ticket_events,0.11,-0.00,0.11,-0.00,0.05,-0.00,0.90,1.00,0.50
credential_validation_events,0.11,0.04,0.11,0.03,0.02,0.00,0.31,0.50,1.00



=== HIGH-VOLUME ROWS WITH NEARLY EQUAL LOGON/PRIVILEGE COUNTS ===


,user,host,hour_window,successful_logons,failed_logons,logoffs,explicit_credential_events,privilege_events,process_starts,kerberos_tgt_events,kerberos_service_ticket_events,credential_validation_events,logon_privilege_difference,logon_logoff_difference,high_logon_volume
439403,Comp916004$,Comp916004,11,90954.0,29.0,89213.0,188.0,90952.0,11407.0,0.0,0.0,0.0,2.0,1741.0,True
127933,Comp916004$,Comp916004,0,39002.0,29.0,37841.0,29.0,39000.0,11453.0,0.0,0.0,0.0,2.0,1161.0,True
311633,AppService,Comp894088,11,10332.0,0.0,10630.0,0.0,10331.0,3.0,0.0,0.0,0.0,1.0,298.0,True
434580,AppService,Comp698972,11,10328.0,0.0,10144.0,0.0,10326.0,6.0,0.0,0.0,0.0,2.0,184.0,True
289738,AppService,Comp894088,10,9959.0,0.0,9553.0,0.0,9958.0,24.0,0.0,0.0,0.0,1.0,406.0,True
263511,AppService,Comp894088,9,9709.0,0.0,9537.0,0.0,9710.0,12.0,0.0,0.0,0.0,1.0,172.0,True
239292,AppService,Comp894088,8,7791.0,0.0,8162.0,0.0,7790.0,39.0,0.0,0.0,0.0,1.0,371.0,True
95232,AppService,Comp698972,7,7219.0,0.0,7290.0,0.0,7220.0,21.0,0.0,0.0,0.0,1.0,71.0,True
155364,Scanner,EnterpriseAppServer,2,6651.0,0.0,6652.0,0.0,6649.0,63.0,0.0,0.0,0.0,2.0,1.0,True
218207,AppService,Comp894088,7,6273.0,0.0,6821.0,0.0,6274.0,12.0,0.0,0.0,0.0,1.0,548.0,True



=== TOP 20 USER-HOST PAIRS ===


,,windows,max_successful_logons,max_privilege_events,max_logoffs
user,host,,,,
Comp916004$,Comp916004,13,126251.0,126270.0,127602.0
AppService,EnterpriseAppServer,13,31822.0,16624.0,32114.0
User897275,Comp897367,13,21219.0,0.0,21450.0
ActiveDirectory$,ActiveDirectory,13,15305.0,13562.0,15302.0
Comp916004$,ActiveDirectory,13,13988.0,13945.0,13804.0
User927159,ActiveDirectory,13,12646.0,0.0,12639.0
Administrator,ActiveDirectory,13,12086.0,0.0,12100.0
Comp915658$,ActiveDirectory,13,11980.0,0.0,11987.0
Scanner,EnterpriseAppServer,12,10852.0,10872.0,10871.0



Rows flagged for data-quality review: 694
Share of dataset: 0.128%

V27 status: diagnostics only; original dataset remains unchanged.


In [ ]:

import numpy as np
import pandas as pd

pairs = [
    ("successful_logons", "logoffs"),
    ("successful_logons", "privilege_events"),
    ("kerberos_tgt_events", "kerberos_service_ticket_events"),
]

print("=== EXACT FEATURE RELATIONSHIP CHECK ===")

for a, b in pairs:
    x = df[a]
    y = df[b]
    diff = x - y

    print(f"\n{a} vs {b}")
    print("Exact matches:", int((x == y).sum()))
    print("Exact match rate:", f"{(x == y).mean():.2%}")
    print("Mean absolute difference:", round(diff.abs().mean(), 3))
    print("Maximum absolute difference:", round(diff.abs().max(), 3))
    print("Correlation:", round(x.corr(y), 5))

    print("Most common differences:")
    display(diff.value_counts().head(10).rename_axis("difference").to_frame("rows"))

print("\n=== DATASET WINDOW DISTRIBUTION ===")
print("Hour windows:", sorted(df["hour_window"].unique().tolist()))
print("Rows per window:")
display(df.groupby("hour_window").size().rename("rows").to_frame())

print("\nNo rows changed. No model trained.")


=== EXACT FEATURE RELATIONSHIP CHECK ===

successful_logons vs logoffs
Exact matches: 428355
Exact match rate: 79.15%
Mean absolute difference: 0.632
Maximum absolute difference: 6237.0
Correlation: 0.99897
Most common differences:


,rows
difference,
0.0,428355
1.0,68748
-1.0,14129
2.0,13781
3.0,4598
-2.0,2630
4.0,1882
5.0,949
-3.0,831



successful_logons vs privilege_events
Exact matches: 247267
Exact match rate: 45.69%
Mean absolute difference: 9.605
Maximum absolute difference: 21219.0
Correlation: 0.95486
Most common differences:


,rows
difference,
0.0,247267
1.0,76248
2.0,26520
5.0,13475
4.0,11177
3.0,8821
18.0,8589
6.0,8360
19.0,8215



kerberos_tgt_events vs kerberos_service_ticket_events
Exact matches: 432426
Exact match rate: 79.90%
Mean absolute difference: 1.174
Maximum absolute difference: 7086.0
Correlation: 0.90487
Most common differences:


,rows
difference,
0.0,432426
-1.0,40696
-2.0,18675
-3.0,9618
-4.0,5510
-8.0,4344
-7.0,4266
-9.0,3784
-6.0,3719



=== DATASET WINDOW DISTRIBUTION ===
Hour windows: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Rows per window:


,rows
hour_window,
0,35380
1,29587
2,29665
3,29350
4,32627
5,33154
6,39992
7,52123
8,58038



No rows changed. No model trained.


In [ ]:

from pathlib import Path
import json
import pandas as pd
import numpy as np

BASE = Path("/content/drive/MyDrive/CyberDNA_LANL")
OUT = BASE / "v27_validation"
OUT.mkdir(parents=True, exist_ok=True)

FEATURES = [
    "successful_logons", "failed_logons", "logoffs",
    "explicit_credential_events", "privilege_events",
    "process_starts", "kerberos_tgt_events",
    "kerberos_service_ticket_events",
    "credential_validation_events",
]

# Feature statistics
stats = df[FEATURES].describe(
    percentiles=[0.5, 0.9, 0.95, 0.99, 0.999]
).T
stats.to_csv(OUT / "feature_statistics.csv")

# Correlation matrix
df[FEATURES].corr().to_csv(OUT / "feature_correlations.csv")

# Per-window population
df.groupby("hour_window").agg(
    rows=("user", "size"),
    unique_users=("user", "nunique"),
    unique_hosts=("host", "nunique"),
).to_csv(OUT / "window_distribution.csv")

# Initial review candidates, not confirmed errors
review_mask = (
    (df["successful_logons"] > 1000)
    | (df["failed_logons"] > 100)
    | (df["explicit_credential_events"] > 1000)
    | (df["privilege_events"] > 1000)
)
df.loc[review_mask].to_csv(
    OUT / "high_count_review_candidates.csv", index=False
)

report = {
    "version": "V27 preliminary data validation",
    "dataset": str(BASE / "lanl_host_aware_aggregated.parquet"),
    "rows": int(len(df)),
    "columns": int(len(df.columns)),
    "features": FEATURES,
    "missing_feature_names": [f for f in FEATURES if f not in df.columns],
    "missing_values_in_features": int(df[FEATURES].isna().sum().sum()),
    "infinite_values_in_features": int(
        np.isinf(df[FEATURES].to_numpy(dtype=float)).sum()
    ),
    "negative_feature_values": int((df[FEATURES] < 0).sum().sum()),
    "duplicate_full_rows": int(df.duplicated().sum()),
    "high_count_review_candidates": int(review_mask.sum()),
    "review_candidate_fraction": float(review_mask.mean()),
    "hour_windows": sorted(int(x) for x in df["hour_window"].unique()),
    "feature_correlations": {
        "successful_logons_vs_logoffs": float(
            df["successful_logons"].corr(df["logoffs"])
        ),
        "successful_logons_vs_privilege_events": float(
            df["successful_logons"].corr(df["privilege_events"])
        ),
        "kerberos_tgt_vs_service_tickets": float(
            df["kerberos_tgt_events"].corr(
                df["kerberos_service_ticket_events"]
            )
        ),
    },
    "status": "Preliminary; source event mapping still requires verification",
    "model_trained": False,
}

with open(OUT / "v27_validation_report.json", "w") as f:
    json.dump(report, f, indent=2)

print("Saved V27 preliminary validation artifacts:")
for path in sorted(OUT.iterdir()):
    print(path.name, f"({path.stat().st_size:,} bytes)")

print("\nStatus:", report["status"])


Saved V27 preliminary validation artifacts:
feature_correlations.csv (1,875 bytes)
feature_statistics.csv (982 bytes)
high_count_review_candidates.csv (51,821 bytes)
v27_validation_report.json (1,123 bytes)
window_distribution.csv (293 bytes)

Status: Preliminary; source event mapping still requires verification


In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/CyberDNA_LANL")
keywords = ("lanl", "aggregat", "feature", "v25", "v26", "dataset")

matches = []

for path in ROOT.rglob("*"):
    if path.is_file() and path.suffix.lower() in {".py", ".ipynb"}:
        if any(word in path.name.lower() for word in keywords):
            matches.append(path)

print(f"Found {len(matches)} possible source files:\n")

for path in matches[:100]:
    print(path)

Found 0 possible source files:



In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive")

extensions = {".ipynb", ".py"}
matches = []

for path in ROOT.rglob("*"):
    if path.is_file() and path.suffix.lower() in extensions:
        matches.append(path)

print(f"Found {len(matches)} notebooks/scripts.\n")

for path in matches[:150]:
    print(path)

if len(matches) > 150:
    print(f"\nShowing first 150 of {len(matches)} files.")

Found 4 notebooks/scripts.

/content/drive/MyDrive/Colab Notebooks/ml_MAGIC.ipynb
/content/drive/MyDrive/Colab Notebooks/ml_Magic.ipynb
/content/drive/MyDrive/Colab Notebooks/learn_ML.ipynb
/content/drive/MyDrive/Colab Notebooks/CyberDNA_Model_Training.ipynb


In [ ]:
parquet = pd.read_parquet(
    "/content/drive/MyDrive/CyberDNA_LANL/lanl_host_aware_aggregated.parquet"
)

parquet_keys = set(
    map(tuple, parquet[["user", "host", "hour_window"]]
        .itertuples(index=False, name=None))
)

missing_from_parquet = affected_keys - parquet_keys

print("Affected keys absent from Parquet:", len(missing_from_parquet))
print("Examples:")
for key in list(missing_from_parquet)[:15]:
    print(key)

Affected keys absent from Parquet: 0
Examples:


In [ ]:
comparison = (
    parquet[["user", "host", "hour_window", "process_ends"]]
    .merge(
        pd.DataFrame(list(affected_keys), columns=["user", "host", "hour_window"]),
        on=["user", "host", "hour_window"],
        how="inner",
    )
    .merge(
        other_totals,
        on=["user", "host", "hour_window"],
        how="left",
    )
)

comparison["other_batches_total"] = (
    comparison["other_batches_total"].fillna(0)
)
comparison["matches"] = (
    comparison["process_ends"] == comparison["other_batches_total"]
)

print("Affected keys checked:", len(comparison))
print("Matching:", int(comparison["matches"].sum()))
print("Mismatching:", int((~comparison["matches"]).sum()))

if (~comparison["matches"]).any():
    print(
        comparison.loc[~comparison["matches"]]
        .head(20)
        .to_string(index=False)
    )

Affected keys checked: 33041
Matching: 33041
Mismatching: 0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

root = Path("/content/drive/MyDrive/CyberDNA_LANL")
print("Project folder exists:", root.exists())
print("Project files:", len(list(root.iterdir())) if root.exists() else 0)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folder exists: True
Project files: 8


In [ ]:

import subprocess
import sys
from pathlib import Path

folder = Path(
    "/content/drive/MyDrive/CyberDNA_LANL/v28_preprocessing"
)

result = subprocess.run(
    [
        sys.executable, "-m", "unittest",
        "test_v28_preprocessing", "-v"
    ],
    cwd=str(folder),
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)

print("Exit code:", result.returncode)



test_feature_order (test_v28_preprocessing.TestV28Preprocessing.test_feature_order) ... ok
test_infinity_rejected (test_v28_preprocessing.TestV28Preprocessing.test_infinity_rejected) ... /content/drive/MyDrive/CyberDNA_LANL/v28_preprocessing/test_v28_preprocessing.py:36: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'inf' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  frame.loc[0, FEATURES[0]] = float("inf")
ok
test_missing_feature_rejected (test_v28_preprocessing.TestV28Preprocessing.test_missing_feature_rejected) ... ok
test_nan_rejected (test_v28_preprocessing.TestV28Preprocessing.test_nan_rejected) ... ok
test_negative_count_rejected (test_v28_preprocessing.TestV28Preprocessing.test_negative_count_rejected) ... ok
test_valid_counts (test_v28_preprocessing.TestV28Preprocessing.test_valid_counts) ... ok

---------------------------------------------------------

In [ ]:

from pathlib import Path

folder = Path(
    "/content/drive/MyDrive/CyberDNA_LANL/v28_preprocessing"
)
test_file = folder / "test_v28_preprocessing.py"

source = test_file.read_text()

old = '''        frame = self.frame.copy()
        frame.loc[0, FEATURES[0]] = float("inf")'''

new = '''        frame = self.frame.copy()
        frame[FEATURES[0]] = frame[FEATURES[0]].astype("float64")
        frame.loc[0, FEATURES[0]] = float("inf")'''

if old not in source:
    print("Expected code not found. No changes made.")
else:
    test_file.write_text(source.replace(old, new, 1))
    print("Infinity test updated.")


Expected code not found. No changes made.


In [ ]:

from pathlib import Path

test_file = Path(
    "/content/drive/MyDrive/CyberDNA_LANL/v28_preprocessing/test_v28_preprocessing.py"
)

for number, line in enumerate(test_file.read_text().splitlines(), start=1):
    if 25 <= number <= 45:
        print(f"{number:3}: {line}")


 25:         with self.assertRaises(ValueError):
 26:             preprocess(frame)
 27: 
 28:     def test_nan_rejected(self):
 29:         frame = self.valid_frame()
 30:         frame.loc[0, FEATURES[0]] = float("nan")
 31:         with self.assertRaises(ValueError):
 32:             preprocess(frame)
 33: 
 34:     def test_infinity_rejected(self):
 35:         frame = self.valid_frame()
 36:         frame.loc[0, FEATURES[0]] = float("inf")
 37:         with self.assertRaises(ValueError):
 38:             preprocess(frame)
 39: 
 40: if __name__ == "__main__":
 41:     unittest.main()


In [ ]:

from pathlib import Path

test_file = Path(
    "/content/drive/MyDrive/CyberDNA_LANL/v28_preprocessing/test_v28_preprocessing.py"
)

source = test_file.read_text()

old = '''    def test_infinity_rejected(self):
        frame = self.valid_frame()
        frame.loc[0, FEATURES[0]] = float("inf")'''

new = '''    def test_infinity_rejected(self):
        frame = self.valid_frame()
        frame[FEATURES[0]] = frame[FEATURES[0]].astype("float64")
        frame.loc[0, FEATURES[0]] = float("inf")'''

if old not in source:
    print("Expected function not found. No changes made.")
else:
    test_file.write_text(source.replace(old, new, 1))
    print("Fixed the infinity test warning.")


Fixed the infinity test warning.


In [ ]:

import subprocess
import sys
from pathlib import Path

folder = Path(
    "/content/drive/MyDrive/CyberDNA_LANL/v28_preprocessing"
)

result = subprocess.run(
    [sys.executable, "-m", "unittest", "test_v28_preprocessing", "-v"],
    cwd=str(folder),
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

print("Exit code:", result.returncode)



test_feature_order (test_v28_preprocessing.TestV28Preprocessing.test_feature_order) ... ok
test_infinity_rejected (test_v28_preprocessing.TestV28Preprocessing.test_infinity_rejected) ... ok
test_missing_feature_rejected (test_v28_preprocessing.TestV28Preprocessing.test_missing_feature_rejected) ... ok
test_nan_rejected (test_v28_preprocessing.TestV28Preprocessing.test_nan_rejected) ... ok
test_negative_count_rejected (test_v28_preprocessing.TestV28Preprocessing.test_negative_count_rejected) ... ok
test_valid_counts (test_v28_preprocessing.TestV28Preprocessing.test_valid_counts) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.020s

OK

Exit code: 0


In [ ]:

from pathlib import Path
import json

folder = Path(
    "/content/drive/MyDrive/CyberDNA_LANL/v28_preprocessing"
)

print("=== V28 FILES ===")
for path in sorted(folder.iterdir()):
    if path.is_file():
        print(f"{path.name} ({path.stat().st_size:,} bytes)")

report_path = folder / "v28_completion_report.json"
if report_path.exists():
    print("\n=== COMPLETION REPORT ===")
    print(json.dumps(json.loads(report_path.read_text()), indent=2))


=== V28 FILES ===
test_v28_preprocessing.py (1,396 bytes)
v25_artifact_hashes.json (312 bytes)
v28_completion_report.json (861 bytes)
v28_preprocessing.py (1,833 bytes)
v28_preprocessing_plan.json (1,478 bytes)

=== COMPLETION REPORT ===
{
  "version": "V28",
  "created_utc": "2026-10-22T14:51:12.760480+00:00",
  "dataset": "lanl_host_aware_aggregated.parquet",
  "rows_validated": 541209,
  "feature_count": 9,
  "feature_order_matches_v25": true,
  "dataset_validation_passed": true,
  "module": "v28_preprocessing.py",
  "test_file": "test_v28_preprocessing.py",
  "transformation": "None; input validation and ordering only",
  "v25_score_compatibility_verified": false,
  "status": "PREPROCESSING_VALIDATION_COMPLETE_COMPATIBILITY_UNVERIFIED",
  "limitations": [
    "Original V25 preprocessing could not be recovered.",
    "The new module does not reproduce V25 score generation.",
    "Do not connect this module to V25 inference until compatibility is verified.",
    "Source event-to-feat

In [ ]:

from pathlib import Path
import json

root = Path("/content/drive/MyDrive/CyberDNA_LANL")

files = [
    root / "v25_final" / "cyberdna_v25_config.json",
    root / "v28_preprocessing" / "v25_artifact_hashes.json",
    root / "v28_preprocessing" / "v28_preprocessing_plan.json",
]

for path in files:
    print(f"\n{'=' * 65}\n{path.name}\n{'=' * 65}")
    if not path.exists():
        print("FILE NOT FOUND")
        continue

    try:
        data = json.loads(path.read_text())
        print(json.dumps(data, indent=2))
    except Exception as exc:
        print(f"Could not read JSON: {exc}")



cyberdna_v25_config.json
{
  "features": [
    "successful_logons",
    "failed_logons",
    "logoffs",
    "explicit_credential_events",
    "privilege_events",
    "process_starts",
    "kerberos_tgt_events",
    "kerberos_service_ticket_events",
    "credential_validation_events"
  ],
  "scales": {
    "successful_logons": 3.4339872044851463,
    "failed_logons": 0.1,
    "logoffs": 3.4339872044851463,
    "explicit_credential_events": 1.0986122886681096,
    "privilege_events": 1.0986122886681096,
    "process_starts": 3.367295829986474,
    "kerberos_tgt_events": 1.6094379124341003,
    "kerberos_service_ticket_events": 2.639057329615259,
    "credential_validation_events": 1.9459101490553132
  },
  "thresholds": {
    "model_score_p99": 0.10532385168133324,
    "model_score_p995": 0.1226613500301712,
    "increase_score_p995": 0.6601691370620053,
    "decrease_score_p995": 0.6705142140817397,
    "calibration_hours": [
      8,
      9
    ],
    "training_hours": [
      1,
   

In [ ]:

from pathlib import Path
import re

root = Path("/content/drive/MyDrive/CyberDNA_LANL")

terms = re.compile(
    r"IsolationForest|cyberdna_v25|model_score_p99|"
    r"successful_logons|RobustScaler|StandardScaler|"
    r"MinMaxScaler|log1p|decision_function|fit_transform",
    re.IGNORECASE,
)

skip_dirs = {
    ".git", ".venv", "venv", "__pycache__",
    "node_modules"
}

matches = []
file_count = 0

for path in root.rglob("*"):
    if not path.is_file():
        continue
    if any(part in skip_dirs for part in path.parts):
        continue
    if path.suffix.lower() not in {".py", ".ipynb"}:
        continue

    file_count += 1

    try:
        content = path.read_text(errors="ignore")
    except Exception:
        continue

    found = sorted({
        term for term in [
            "IsolationForest", "cyberdna_v25", "model_score_p99",
            "successful_logons", "RobustScaler", "StandardScaler",
            "MinMaxScaler", "log1p", "decision_function",
            "fit_transform"
        ]
        if re.search(re.escape(term), content, re.IGNORECASE)
    })

    if found:
        matches.append((path, found))

print("Python/notebook files scanned:", file_count)
print("Files with relevant references:", len(matches))

for path, found in matches[:100]:
    print(f"\n{path}")
    print("  Found:", ", ".join(found))


Python/notebook files scanned: 2
Files with relevant references: 1

/content/drive/MyDrive/CyberDNA_LANL/v28_preprocessing/v28_preprocessing.py
  Found: successful_logons


In [ ]:

import json
from pathlib import Path

notebook_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/CyberDNA_Model_Training.ipynb"
)

with notebook_path.open("r", encoding="utf-8") as f:
    notebook = json.load(f)

terms = [
    "IsolationForest",
    "successful_logons",
    "decision_function",
    "fit_transform",
    "StandardScaler",
    "RobustScaler",
    "MinMaxScaler",
    "log1p",
    "scales",
    "model_score_p99",
]

print("Notebook:", notebook_path.name)
print("Total cells:", len(notebook.get("cells", [])))

for i, cell in enumerate(notebook.get("cells", [])):
    source = "".join(cell.get("source", []))
    matched = [term for term in terms if term.lower() in source.lower()]

    if matched:
        print(f"\n{'=' * 60}")
        print(f"CELL {i} | Type: {cell.get('cell_type')}")
        print("Matches:", ", ".join(matched))
        print("-" * 60)
        print(source[:7000])


Notebook: CyberDNA_Model_Training.ipynb
Total cells: 62

CELL 4 | Type: code
Matches: successful_logons
------------------------------------------------------------
print('=== PART 1.2: CLASS SUPPORT & DISJOINT-USER SAMPLE AUDIT ===')
if data_path.exists():
    # Standardize column naming
    df_eval = df_eval.rename(columns={'failed_logons': 'failed_ins' if 'failed_ins' in df_eval.columns else 'failed_logins'})
    df_eval['is_computer'] = df_eval['user'].str.endswith('$').astype(int)
    df_eval['total_auth_attempts'] = df_eval['successful_logons'] + df_eval['failed_logins']
    df_eval['login_failure_rate'] = np.where(df_eval['total_auth_attempts'] > 0, df_eval['failed_logins'] / df_eval['total_auth_attempts'], 0.0)

    # Generate behavior classes
    conds = [
        (df_eval['failed_logins'] >= 50) & (df_eval['login_failure_rate'] > 0.8),
        (df_eval['privilege_events'] >= 100) & (df_eval['is_computer'] == 0)
    ]
    df_eval['behavior_class'] = np.select(conds, ['high_fai

In [ ]:

import json
from pathlib import Path

root = Path("/content/drive/MyDrive/CyberDNA_LANL")
config_path = root / "v25_final" / "cyberdna_v25_config.json"
model_path = root / "v25_final" / "cyberdna_v25_isolation_forest.joblib"

config = json.loads(config_path.read_text())

print("Feature order:")
for i, feature in enumerate(config["features"], 1):
    print(f"{i}. {feature}")

print("\nSaved feature scales:")
for feature, scale in config["scales"].items():
    print(f"{feature}: {scale}")

print("\nModel settings:")
print("Estimators:", config["n_estimators"])
print("Random state:", config["random_state"])
print("Model file exists:", model_path.exists())
print("Feature count:", len(config["features"]))
print("Scale count:", len(config["scales"]))
print("Feature names unique:", len(set(config["features"])) == len(config["features"]))
print("Feature/scale names match:",
      set(config["features"]) == set(config["scales"]))


Feature order:
1. successful_logons
2. failed_logons
3. logoffs
4. explicit_credential_events
5. privilege_events
6. process_starts
7. kerberos_tgt_events
8. kerberos_service_ticket_events
9. credential_validation_events

Saved feature scales:
successful_logons: 3.4339872044851463
failed_logons: 0.1
logoffs: 3.4339872044851463
explicit_credential_events: 1.0986122886681096
privilege_events: 1.0986122886681096
process_starts: 3.367295829986474
kerberos_tgt_events: 1.6094379124341003
kerberos_service_ticket_events: 2.639057329615259
credential_validation_events: 1.9459101490553132

Model settings:
Estimators: 300
Random state: 2026
Model file exists: True
Feature count: 9
Scale count: 9
Feature names unique: True
Feature/scale names match: True


In [ ]:

from pathlib import Path
import json
import importlib.util
import numpy as np
import pandas as pd

ROOT = Path("/content/drive/MyDrive/CyberDNA_LANL")
V28 = ROOT / "v28_preprocessing"
CONFIG_PATH = ROOT / "v25_final" / "cyberdna_v25_config.json"

assert CONFIG_PATH.exists(), f"Missing V25 config: {CONFIG_PATH}"
V28.mkdir(parents=True, exist_ok=True)

config = json.loads(CONFIG_PATH.read_text())
FEATURES = config["features"]
SCALES = {k: float(v) for k, v in config["scales"].items()}

assert len(FEATURES) == 9 and len(set(FEATURES)) == 9
assert set(FEATURES) == set(SCALES)
assert all(np.isfinite(v) and v > 0 for v in SCALES.values())

module_code = r'''
import numpy as np
import pandas as pd

FEATURES = [
    "successful_logons",
    "failed_logons",
    "logoffs",
    "explicit_credential_events",
    "privilege_events",
    "process_starts",
    "kerberos_tgt_events",
    "kerberos_service_ticket_events",
    "credential_validation_events",
]
METADATA = ["user", "host", "hour_window"]

def preprocess(frame, scales):
    """Reproduce V25 hourly-delta preprocessing.

    Returns all rows sorted by user, host and hour_window, with:
      - original metadata
      - V25-transformed feature columns
      - scoreable: True only for consecutive hours within a group

    Pass the saved V25 config scales. Never fit scales here.
    """
    missing = [c for c in METADATA + FEATURES if c not in frame.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    if set(scales) != set(FEATURES):
        raise ValueError("Saved scale names must match the nine V25 features exactly")

    scales = {f: float(scales[f]) for f in FEATURES}
    if any(not np.isfinite(v) or v <= 0 for v in scales.values()):
        raise ValueError("All saved feature scales must be finite and positive")

    df = frame[METADATA + FEATURES].copy()

    if df[METADATA].isna().any().any():
        raise ValueError("Metadata contains missing values")

    for f in FEATURES:
        df[f] = pd.to_numeric(df[f], errors="raise")

    raw = df[FEATURES].to_numpy(dtype=np.float64)
    if not np.isfinite(raw).all():
        raise ValueError("Raw event counts contain NaN or infinity")
    if (raw < 0).any():
        raise ValueError("Raw event counts cannot be negative")

    hours = pd.to_numeric(df["hour_window"], errors="raise")
    if not np.isfinite(hours.to_numpy(dtype=float)).all():
        raise ValueError("hour_window must contain finite integers")
    if not np.equal(hours.to_numpy(dtype=float),
                    np.floor(hours.to_numpy(dtype=float))).all():
        raise ValueError("hour_window must contain integers")
    df["hour_window"] = hours.astype(int)

    if not df["hour_window"].between(0, 12).all():
        raise ValueError("hour_window must be between 0 and 12")

    if df.duplicated(["user", "host", "hour_window"]).any():
        raise ValueError("Duplicate user/host/hour_window keys are ambiguous")

    df = df.sort_values(
        ["user", "host", "hour_window"], kind="mergesort"
    ).reset_index(drop=True)

    groups = df.groupby(["user", "host"], sort=False)
    previous = groups[FEATURES].shift(1)
    previous_hour = groups["hour_window"].shift(1)
    gap = df["hour_window"] - previous_hour
    scoreable = gap.eq(1)

    delta = df[FEATURES] - previous
    signed_log = np.sign(delta) * np.log1p(np.abs(delta))

    X = signed_log.copy()
    for f in FEATURES:
        X[f] = (X[f] / scales[f]).clip(-5, 5)

    X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    result = df[METADATA].copy()
    result[FEATURES] = X[FEATURES].astype(np.float64)
    result["scoreable"] = scoreable.to_numpy(dtype=bool)
    return result
'''

module_path = V28 / "v28_preprocessing.py"
module_path.write_text(module_code)

spec = importlib.util.spec_from_file_location(
    "v28_preprocessing", module_path
)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

# ---- Tests using the actual saved V25 scales ----
def sample_frame():
    rows = []
    for hour, count in [(0, 2), (1, 5), (3, 8)]:
        row = {"user": "alice", "host": "pc1", "hour_window": hour}
        row.update({f: 0 for f in FEATURES})
        row["successful_logons"] = count
        rows.append(row)
    row = {"user": "bob", "host": "pc2", "hour_window": 1}
    row.update({f: 0 for f in FEATURES})
    row["successful_logons"] = 10
    rows.append(row)
    return pd.DataFrame(rows)

sample = sample_frame()
out = mod.preprocess(sample, SCALES)

# Feature order and finite output
assert list(out.columns[:3]) == ["user", "host", "hour_window"]
assert list(out.columns[3:12]) == FEATURES
assert np.isfinite(out[FEATURES].to_numpy()).all()

# Alice hour 0 is first in group; hour 1 is consecutive; hour 3 has a gap.
alice = out[out["user"] == "alice"].set_index("hour_window")
assert not bool(alice.loc[0, "scoreable"])
assert bool(alice.loc[1, "scoreable"])
assert not bool(alice.loc[3, "scoreable"])

# V25 formula for successful_logons: (5 - 2) -> log1p(3) / saved scale.
expected = np.log1p(3) / SCALES["successful_logons"]
assert np.isclose(
    alice.loc[1, "successful_logons"], np.clip(expected, -5, 5)
)

# Negative changes preserve their sign.
neg = sample_frame()
neg.loc[neg["hour_window"] == 1, "successful_logons"] = 1
neg_out = mod.preprocess(neg, SCALES)
alice_neg = neg_out[
    (neg_out["user"] == "alice") & (neg_out["hour_window"] == 1)
].iloc[0]
assert alice_neg["successful_logons"] < 0

# User/host groups must not borrow another group's previous event.
bob = out[out["user"] == "bob"].iloc[0]
assert not bool(bob["scoreable"])

# Invalid inputs must fail.
for bad_value in [-1, np.nan, np.inf]:
    bad = sample_frame()
    bad.loc[0, "successful_logons"] = bad_value
    try:
        mod.preprocess(bad, SCALES)
        raise AssertionError(f"Invalid value {bad_value} was accepted")
    except ValueError:
        pass

# Missing features and mismatched scale keys must fail.
try:
    mod.preprocess(sample.drop(columns=[FEATURES[0]]), SCALES)
    raise AssertionError("Missing feature was accepted")
except ValueError:
    pass

try:
    mod.preprocess(sample, {**SCALES, "wrong_feature": 1.0})
    raise AssertionError("Incorrect scale keys were accepted")
except ValueError:
    pass

print("V28 module written:", module_path)
print("Saved V25 scales loaded:", len(SCALES))
print("All preprocessing tests passed.")
print("Scoreable test rows:", int(out["scoreable"].sum()))
print("Total test rows:", len(out))
print("\nTransformed test example:")
print(out[["user", "host", "hour_window", "scoreable",
           "successful_logons"]].to_string(index=False))


V28 module written: /content/drive/MyDrive/CyberDNA_LANL/v28_preprocessing/v28_preprocessing.py
Saved V25 scales loaded: 9
All preprocessing tests passed.
Scoreable test rows: 1
Total test rows: 4

Transformed test example:
 user host  hour_window  scoreable  successful_logons
alice  pc1            0      False           0.000000
alice  pc1            1       True           0.403698
alice  pc1            3      False           0.403698
  bob  pc2            1      False           0.000000


/tmp/ipykernel_1425/1799658352.py:172: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'inf' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  bad.loc[0, "successful_logons"] = bad_value


In [ ]:

import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
import importlib.util

ROOT = Path("/content/drive/MyDrive/CyberDNA_LANL")
V25 = ROOT / "v25_final"
V28 = ROOT / "v28_preprocessing"

config = json.loads((V25 / "cyberdna_v25_config.json").read_text())
scales = config["scales"]
features = config["features"]

data_path = ROOT / "lanl_host_aware_aggregated.parquet"
model_path = V25 / "cyberdna_v25_isolation_forest.joblib"
scores_path = V25 / "cyberdna_v25_scores.csv"

assert data_path.exists(), f"Missing dataset: {data_path}"
assert model_path.exists(), f"Missing model: {model_path}"
assert scores_path.exists(), f"Missing saved scores: {scores_path}"

spec = importlib.util.spec_from_file_location(
    "v28_preprocessing", V28 / "v28_preprocessing.py"
)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

df = pd.read_parquet(data_path)
processed = mod.preprocess(df, scales)
saved = pd.read_csv(scores_path)
model = joblib.load(model_path)

print("Saved-score columns:", saved.columns.tolist())
print("Saved-score rows:", len(saved))
print("Dataset rows:", len(df))
print("V28 scoreable rows:", int(processed["scoreable"].sum()))
print("\nSaved score preview:")
print(saved.head(3).to_string(index=False))

# Only score rows with consecutive-hour history, as V25 did.
valid = processed.loc[processed["scoreable"]].copy()
X = valid[features].to_numpy(dtype=np.float64)
computed_scores = -model.decision_function(X)

print("\nComputed scores:", len(computed_scores))
print("Computed score range:",
      float(np.min(computed_scores)), float(np.max(computed_scores)))

# We inspect the saved schema before attempting an automatic key-based match.
possible_keys = ["user", "host", "hour_window"]
print("\nAvailable join keys:",
      [c for c in possible_keys if c in saved.columns])

score_columns = [
    c for c in saved.columns
    if "score" in c.lower() or "anomaly" in c.lower()
]
print("Possible saved score columns:", score_columns)
print("\nDo not overwrite any V25 files.")


Saved-score columns: ['user', 'host', 'hour_window', 'anomaly_score', 'increase_score', 'decrease_score', 'split', 'model_candidate_p99', 'model_candidate_p995', 'increase_candidate_p995', 'decrease_candidate_p995']
Saved-score rows: 541209
Dataset rows: 541209
V28 scoreable rows: 430195

Saved score preview:
            user            host  hour_window  anomaly_score  increase_score  decrease_score    split  model_candidate_p99  model_candidate_p995  increase_candidate_p995  decrease_candidate_p995
ActiveDirectory$ ActiveDirectory            0            NaN        0.000000        0.000000 unscored                False                 False                    False                    False
ActiveDirectory$ ActiveDirectory            1       0.156904        0.194504        0.923219    train                False                 False                    False                    False
ActiveDirectory$ ActiveDirectory            2       0.151245        1.097993        0.087550    train   

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(



Computed scores: 430195
Computed score range: -0.18434047135328202 0.22976135616825055

Available join keys: ['user', 'host', 'hour_window']
Possible saved score columns: ['anomaly_score', 'increase_score', 'decrease_score']

Do not overwrite any V25 files.


In [ ]:

import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
import importlib.util

ROOT = Path("/content/drive/MyDrive/CyberDNA_LANL")
V25 = ROOT / "v25_final"
V28 = ROOT / "v28_preprocessing"

config = json.loads((V25 / "cyberdna_v25_config.json").read_text())
features = config["features"]
scales = config["scales"]

spec = importlib.util.spec_from_file_location(
    "v28_preprocessing", V28 / "v28_preprocessing.py"
)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

raw = pd.read_parquet(ROOT / "lanl_host_aware_aggregated.parquet")
saved = pd.read_csv(V25 / "cyberdna_v25_scores.csv")
model = joblib.load(V25 / "cyberdna_v25_isolation_forest.joblib")

processed = mod.preprocess(raw, scales)
valid = processed.loc[processed["scoreable"]].copy()

# Preserve feature names to match the model's original input format.
X = valid[features].astype(np.float64)
computed = valid[["user", "host", "hour_window"]].copy()
computed["anomaly_score"] = -model.decision_function(X)

# Reproduce V25 directional scores from transformed feature values.
computed["increase_score"] = X.clip(lower=0).mean(axis=1).to_numpy()
computed["decrease_score"] = (-X.clip(upper=0)).mean(axis=1).to_numpy()

keys = ["user", "host", "hour_window"]
score_cols = ["anomaly_score", "increase_score", "decrease_score"]

assert not saved.duplicated(keys).any(), "Duplicate keys in saved score file"
assert not computed.duplicated(keys).any(), "Duplicate keys in computed scores"

comparison = saved[keys + score_cols].merge(
    computed[keys + score_cols],
    on=keys,
    how="outer",
    suffixes=("_saved", "_new"),
    indicator=True,
    validate="one_to_one",
)

print("===== KEY MATCHING =====")
print(comparison["_merge"].value_counts(dropna=False).to_string())

matched = comparison[comparison["_merge"] == "both"].copy()
print("\nMatched rows:", len(matched))
print("Saved scored rows:", int(saved["anomaly_score"].notna().sum()))

print("\n===== NUMERICAL COMPARISON =====")
all_pass = True
for col in score_cols:
    a = matched[f"{col}_saved"].to_numpy(dtype=float)
    b = matched[f"{col}_new"].to_numpy(dtype=float)
    finite = np.isfinite(a) & np.isfinite(b)

    if not finite.any():
        print(f"{col}: no finite matched values")
        all_pass = False
        continue

    diff = np.abs(a[finite] - b[finite])
    max_diff = float(diff.max())
    mean_diff = float(diff.mean())
    close = np.isclose(a[finite], b[finite], rtol=1e-7, atol=1e-9)
    pass_col = bool(close.all())
    all_pass &= pass_col

    print(
        f"{col}: compared={int(finite.sum())}, "
        f"max_abs_diff={max_diff:.12g}, "
        f"mean_abs_diff={mean_diff:.12g}, "
        f"within_tolerance={int(close.sum())}/{len(close)}, "
        f"PASS={pass_col}"
    )

print("\n===== FINAL RESULT =====")
print("V25 score reproduction:", "PASS" if all_pass else "NOT YET VERIFIED")
print("V25 artifacts modified: No")

if not all_pass:
    print(
        "\nDo not integrate the model yet. Send this output so we can "
        "identify whether the mismatch is in preprocessing or scoring."
    )


===== KEY MATCHING =====
_merge
both          430195
left_only     111014
right_only         0

Matched rows: 430195
Saved scored rows: 430195

===== NUMERICAL COMPARISON =====
anomaly_score: compared=430195, max_abs_diff=9.98550200859e-17, mean_abs_diff=3.94271362024e-17, within_tolerance=430195/430195, PASS=True
increase_score: compared=430195, max_abs_diff=4.4408920985e-16, mean_abs_diff=1.74488192826e-17, within_tolerance=430195/430195, PASS=True
decrease_score: compared=430195, max_abs_diff=2.22044604925e-16, mean_abs_diff=1.9019427623e-17, within_tolerance=430195/430195, PASS=True

===== FINAL RESULT =====
V25 score reproduction: PASS
V25 artifacts modified: No


In [ ]:

from pathlib import Path
import json
from datetime import datetime, timezone

ROOT = Path("/content/drive/MyDrive/CyberDNA_LANL")
V28 = ROOT / "v28_preprocessing"
report_path = V28 / "v28_completion_report.json"

assert report_path.exists(), f"Missing report: {report_path}"

report = json.loads(report_path.read_text())

report.update({
    "preprocessing_implementation": "V25-compatible hourly-delta transformation",
    "v25_score_compatibility_verified": True,
    "score_reproduction": {
        "matched_rows": 430195,
        "saved_scored_rows": 430195,
        "unscored_rows": 111014,
        "anomaly_score_max_abs_diff": 9.98550200859e-17,
        "increase_score_max_abs_diff": 4.4408920985e-16,
        "decrease_score_max_abs_diff": 2.22044604925e-16,
        "all_scores_within_tolerance": True,
        "tolerance": {
            "relative": 1e-7,
            "absolute": 1e-9
        }
    },
    "v25_artifacts_modified": False,
    "collector_feature_mapping_verified": False,
    "integration_ready": False,
    "remaining_blocker": (
        "Verify Windows collector event-ID mappings, feature definitions, "
        "aggregation windows, and compatibility with the training data."
    ),
    "last_validation_utc": datetime.now(timezone.utc).isoformat()
})

report_path.write_text(json.dumps(report, indent=2) + "\n")

# Read back to confirm the report was written correctly.
verified = json.loads(report_path.read_text())
assert verified["v25_score_compatibility_verified"] is True
assert verified["score_reproduction"]["all_scores_within_tolerance"] is True
assert verified["collector_feature_mapping_verified"] is False
assert verified["integration_ready"] is False

print("V28 completion report updated and verified.")
print("V25 score compatibility: PASS")
print("Collector mapping: STILL NEEDS VALIDATION")
print("Integration readiness: NOT YET CONFIRMED")
print("Report:", report_path)


V28 completion report updated and verified.
V25 score compatibility: PASS
Collector mapping: STILL NEEDS VALIDATION
Integration readiness: NOT YET CONFIRMED
Report: /content/drive/MyDrive/CyberDNA_LANL/v28_preprocessing/v28_completion_report.json


In [ ]:

import json
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "CyberDNA_Model_Training.ipynb"
)
assert path.exists(), f"Notebook not found: {path}"

notebook = json.loads(path.read_text())

keywords = [
    "lanl_host_aware_aggregated",
    "successful_logons",
    "failed_logons",
    "hour_window",
    "kerberos_tgt_events",
    "IsolationForest",
    "decision_function",
    "threshold",
    "quantile",
    "train_test_split",
    "feature",
    "scale",
    "joblib.dump",
    "to_csv",
]

for i, cell in enumerate(notebook.get("cells", [])):
    source = "".join(cell.get("source", []))

    if any(k.lower() in source.lower() for k in keywords):
        print(f"\n{'=' * 20} CELL {i} "
              f"({cell.get('cell_type', 'unknown')}) "
              f"{'=' * 20}\n")
        print(source)



==================== CELL 3 (code) ====================

import numpy as np
import pandas as pd
from pathlib import Path
import json

base_dir = Path('/content/drive/MyDrive/CyberDNA_LANL')
data_path = base_dir / 'lanl_host_aware_aggregated.parquet'

print('=== PART 1.1: BOUNDARY ANALYSIS ===')
if data_path.exists():
    df_eval = pd.read_parquet(data_path)
    print(f'Loaded {len(df_eval):,} aggregated rows.')

    # Check exact hours in dataset
    hours = sorted(df_eval['hour_window'].unique())
    print('Unique Hours in Dataset:', hours)

    # Recalculate 75th percentile split hour
    split_hour = int(df_eval['hour_window'].quantile(0.75))
    print(f'Calculated 75th Percentile Hour: {split_hour}')

    train_hours = df_eval[df_eval['hour_window'] < split_hour]['hour_window'].unique()
    test_hours = df_eval[df_eval['hour_window'] >= split_hour]['hour_window'].unique()
    print(f'Training hours (strictly < {split_hour}): {sorted(train_hours)}')
    print(f'Testing hours (stric

In [ ]:

import json
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "CyberDNA_Model_Training.ipynb"
)
notebook = json.loads(path.read_text())

# Save a focused audit file instead of printing huge cells.
keywords = [
    "lanl_host_aware_aggregated",
    "successful_logons",
    "failed_logons",
    "hour_window",
    "kerberos_tgt_events",
    "IsolationForest",
    "decision_function",
    "threshold",
    "quantile",
    "train_test_split",
    "scale",
    "joblib.dump",
    "to_csv",
]

output = []
for i, cell in enumerate(notebook.get("cells", [])):
    source = "".join(cell.get("source", []))
    matched = [
        k for k in keywords
        if k.lower() in source.lower()
    ]

    if matched:
        output.append(
            f"\n{'='*18} CELL {i} {'='*18}\n"
            f"Matched: {', '.join(matched)}\n"
            f"{source}\n"
        )

out_path = Path(
    "/content/drive/MyDrive/CyberDNA_LANL/"
    "v28_preprocessing/original_notebook_audit.txt"
)
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text("\n".join(output))

print("Audit saved to:", out_path)
print("Matching cells:", len(output))
print("\nCell index and matched keywords:")
for item in output:
    header = item.splitlines()[:2]
    print(" | ".join(header))


Audit saved to: /content/drive/MyDrive/CyberDNA_LANL/v28_preprocessing/original_notebook_audit.txt
Matching cells: 37

Cell index and matched keywords:
 | ================== CELL 3 ==================
 | ================== CELL 4 ==================
 | ================== CELL 5 ==================
 | ================== CELL 6 ==================
 | ================== CELL 7 ==================
 | ================== CELL 11 ==================
 | ================== CELL 12 ==================
 | ================== CELL 13 ==================
 | ================== CELL 16 ==================
 | ================== CELL 17 ==================
 | ================== CELL 18 ==================
 | ================== CELL 19 ==================
 | ================== CELL 22 ==================
 | ================== CELL 23 ==================
 | ================== CELL 24 ==================
 | ================== CELL 25 ==================
 | ================== CELL 26 ==================
 | =================

In [ ]:

from pathlib import Path
import json

root = Path("/content/drive/MyDrive/CyberDNA_LANL")
v25 = root / "v25_final"
v28 = root / "v28_preprocessing"

for name in [
    "cyberdna_v25_isolation_forest.joblib",
    "cyberdna_v25_config.json",
    "cyberdna_v25_thresholds.json",
    "cyberdna_v25_scores.csv",
]:
    path = v25 / name
    print(f"{name}: {'EXISTS' if path.exists() else 'MISSING'}")

print("\nV28 files:")
for name in [
    "v28_preprocessing.py",
    "v25_artifact_hashes.json",
]:
    path = v28 / name
    print(f"{name}: {'EXISTS' if path.exists() else 'MISSING'}")

config_path = v25 / "cyberdna_v25_config.json"
if config_path.exists():
    config = json.loads(config_path.read_text())
    print("\nFeature names:", config.get("feature_names"))
    print("Saved scales available:", bool(config.get("feature_scales")))


cyberdna_v25_isolation_forest.joblib: EXISTS
cyberdna_v25_config.json: EXISTS
cyberdna_v25_thresholds.json: EXISTS
cyberdna_v25_scores.csv: EXISTS

V28 files:
v28_preprocessing.py: EXISTS
v25_artifact_hashes.json: EXISTS

Feature names: None
Saved scales available: False


In [ ]:

import json
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/CyberDNA_LANL/"
    "v25_final/cyberdna_v25_config.json"
)

config = json.loads(path.read_text())

print("Top-level keys:")
print(list(config.keys()))

print("\nConfig contents:")
print(json.dumps(config, indent=2)[:12000])


Top-level keys:
['features', 'scales', 'thresholds', 'random_state', 'n_estimators']

Config contents:
{
  "features": [
    "successful_logons",
    "failed_logons",
    "logoffs",
    "explicit_credential_events",
    "privilege_events",
    "process_starts",
    "kerberos_tgt_events",
    "kerberos_service_ticket_events",
    "credential_validation_events"
  ],
  "scales": {
    "successful_logons": 3.4339872044851463,
    "failed_logons": 0.1,
    "logoffs": 3.4339872044851463,
    "explicit_credential_events": 1.0986122886681096,
    "privilege_events": 1.0986122886681096,
    "process_starts": 3.367295829986474,
    "kerberos_tgt_events": 1.6094379124341003,
    "kerberos_service_ticket_events": 2.639057329615259,
    "credential_validation_events": 1.9459101490553132
  },
  "thresholds": {
    "model_score_p99": 0.10532385168133324,
    "model_score_p995": 0.1226613500301712,
    "increase_score_p995": 0.6601691370620053,
    "decrease_score_p995": 0.6705142140817397,
    "calib

In [ ]:

import inspect
import sys
from pathlib import Path

v28_dir = Path(
    "/content/drive/MyDrive/CyberDNA_LANL/v28_preprocessing"
)
sys.path.insert(0, str(v28_dir))

# Reload the module from the current file.
import importlib
import v28_preprocessing as v28
v28 = importlib.reload(v28)

print("Module file:", v28.__file__)
print("Function signature:", inspect.signature(v28.preprocess))
print("\nFunction source:\n")
print(inspect.getsource(v28.preprocess))


Module file: /content/drive/MyDrive/CyberDNA_LANL/v28_preprocessing/v28_preprocessing.py
Function signature: (frame, scales)

Function source:

def preprocess(frame, scales):
    """Reproduce V25 hourly-delta preprocessing.

    Returns all rows sorted by user, host and hour_window, with:
      - original metadata
      - V25-transformed feature columns
      - scoreable: True only for consecutive hours within a group

    Pass the saved V25 config scales. Never fit scales here.
    """
    missing = [c for c in METADATA + FEATURES if c not in frame.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    if set(scales) != set(FEATURES):
        raise ValueError("Saved scale names must match the nine V25 features exactly")

    scales = {f: float(scales[f]) for f in FEATURES}
    if any(not np.isfinite(v) or v <= 0 for v in scales.values()):
        raise ValueError("All saved feature scales must be finite and positive")

    df = frame[METADATA + 

In [ ]:

import sys
import json
import hashlib
import importlib
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/CyberDNA_LANL")
V25 = ROOT / "v25_final"
V28 = ROOT / "v28_preprocessing"

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

artifact_names = [
    "cyberdna_v25_isolation_forest.joblib",
    "cyberdna_v25_config.json",
    "cyberdna_v25_thresholds.json",
    "cyberdna_v25_scores.csv",
]
before = {n: sha256(V25 / n) for n in artifact_names}

config = json.loads(
    (V25 / "cyberdna_v25_config.json").read_text()
)
features = config["features"]
scales = config["scales"]

sys.path.insert(0, str(V28))
import v28_preprocessing
v28 = importlib.reload(v28_preprocessing)

raw = pd.read_parquet(
    ROOT / "lanl_host_aware_aggregated.parquet"
)
saved = pd.read_csv(V25 / "cyberdna_v25_scores.csv")
model = joblib.load(
    V25 / "cyberdna_v25_isolation_forest.joblib"
)

print("=== 1. Input checks ===")
print("Raw rows:", len(raw))
print("Saved score rows:", len(saved))
print("Features:", len(features))
print("Scale names match:",
      set(features) == set(scales))
print("Model feature count:", getattr(model, "n_features_in_", "unknown"))

print("\n=== 2. Preprocessing ===")
processed = v28.preprocess(raw, scales)
valid = processed.loc[processed["scoreable"]].copy()
print("Processed rows:", len(processed))
print("Scoreable rows:", len(valid))
print("Unscoreable rows:", int((~processed["scoreable"]).sum()))
print("All transformed features finite:",
      bool(np.isfinite(valid[features].to_numpy(dtype=float)).all()))

if getattr(model, "n_features_in_", len(features)) != len(features):
    raise ValueError("Model feature count does not match V25 config.")

print("\n=== 3. Recalculate and compare scores ===")
X = valid[features].astype("float64")
valid["recomputed_score"] = -model.decision_function(X)

keys = ["user", "host", "hour_window"]
comparison = saved[keys + ["anomaly_score"]].merge(
    valid[keys + ["recomputed_score"]],
    on=keys,
    how="inner",
    validate="one_to_one",
)
comparison = comparison.dropna(
    subset=["anomaly_score", "recomputed_score"]
)

print("Matched scored rows:", len(comparison))
if len(comparison) == 0:
    print("RESULT: FAIL — no comparable scores.")
else:
    diff = (
        comparison["anomaly_score"] -
        comparison["recomputed_score"]
    ).abs()
    print("Maximum absolute difference:", float(diff.max()))
    print("Mean absolute difference:", float(diff.mean()))
    print("All differences below 1e-10:",
          bool((diff < 1e-10).all()))

print("\n=== 4. Original V25 artifact integrity ===")
after = {n: sha256(V25 / n) for n in artifact_names}
for n in artifact_names:
    print(n, "UNCHANGED" if before[n] == after[n] else "CHANGED")

print("\nNo model training or artifact writes were performed.")


=== 1. Input checks ===
Raw rows: 541209
Saved score rows: 541209
Features: 9
Scale names match: True
Model feature count: 9

=== 2. Preprocessing ===
Processed rows: 541209
Scoreable rows: 430195
Unscoreable rows: 111014
All transformed features finite: True

=== 3. Recalculate and compare scores ===
Matched scored rows: 430195
Maximum absolute difference: 9.985502008591496e-17
Mean absolute difference: 3.9427136202398264e-17
All differences below 1e-10: True

=== 4. Original V25 artifact integrity ===
cyberdna_v25_isolation_forest.joblib UNCHANGED
cyberdna_v25_config.json UNCHANGED
cyberdna_v25_thresholds.json UNCHANGED
cyberdna_v25_scores.csv UNCHANGED

No model training or artifact writes were performed.
